# Causality and abstraction analysis

Read existing Mediator outputs and QA scores directly from the database. No database initialization, input loading, full Mediator execution, or debug reruns are performed here. Static covariates and visit metadata remain separate configurable inputs. Controlled abstraction-stability experiments are an explicit section at the end.


In [ ]:
from pathlib import Path
import os
import sys
import sqlite3
import subprocess
from functools import lru_cache
import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'analysis':
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'core/mediator.py').exists():
    raise RuntimeError('Open from the Mediator repository or its analysis directory')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# All stored configuration paths are repository-relative. Override for another dataset.
DB_PATH = Path('backend/data/mediator.db')
EXTERNAL_DATA_DIR = Path('../Data Exploration')
ANALYSIS_CODE = Path('analysis')
ANALYSIS_RESULTS = Path(os.environ.get('MEDIATOR_ANALYSIS_OUTPUT', 'backend/data/analysis'))
RESULTS_DIR = ANALYSIS_RESULTS / 'causality'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(RESULTS_DIR / 'matplotlib')

def connect_readonly():
    connection = sqlite3.connect(DB_PATH.resolve().as_uri() + '?mode=ro', uri=True)
    connection.execute('PRAGMA query_only=ON')
    return connection

@lru_cache(maxsize=2)
def _read_database_table(table):
    queries = {
        'output': 'SELECT PatientId, ConceptName, StartDateTime, EndDateTime, Value FROM OutputPatientData',
        'qa': 'SELECT PatientId, PatternName, StartDateTime, EndDateTime, ComplianceType, ComplianceScore FROM PatientQAScores',
    }
    connection = connect_readonly()
    try:
        return pd.read_sql_query(queries[table], connection)
    finally:
        connection.close()

def read_output_data(parse_dates=None, **kwargs):
    frame = _read_database_table('output').copy()
    for column in parse_dates or []:
        frame[column] = pd.to_datetime(frame[column])
    return frame

def read_qa_scores(parse_dates=None, **kwargs):
    frame = _read_database_table('qa').copy()
    for column in parse_dates or []:
        frame[column] = pd.to_datetime(frame[column])
    return frame

def run_analysis(folder, script, *arguments):
    subprocess.run([sys.executable, str(ANALYSIS_CODE / folder / script), *map(str, arguments)],
                   cwd=REPO_ROOT, check=True)


# 🔍 Adherance Analysis

In [ ]:
import pandas as pd
import os
os.environ["MPLCONFIGDIR"] = str(RESULTS_DIR / "matplotlib")

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.interpolate import make_interp_spline
import statsmodels.api as sm

# ------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------
df = read_qa_scores(parse_dates=['StartDateTime', 'EndDateTime'])

# ------------------------------------------------------
# STEP 1: Aggregation (day-then-overall, so a pattern scored many times
# on one day doesn't outweigh a day it was only scored once)
# ------------------------------------------------------
df['Date'] = df['StartDateTime'].dt.date

daily = (
    df.groupby(['PatientId', 'PatternName', 'ComplianceType', 'Date'])['ComplianceScore']
    .mean()
    .reset_index()
)

df_agg = (
    daily.groupby(['PatientId', 'PatternName', 'ComplianceType'])['ComplianceScore']
    .mean()
    .reset_index()
    .rename(columns={'ComplianceScore': 'AvgScore'})
)

span = df.groupby(['PatientId', 'PatternName', 'ComplianceType']).agg(
    MinStartDateTime=('StartDateTime', 'min'),
    MaxEndDateTime=('EndDateTime', 'max'),
).reset_index()

df_agg = df_agg.merge(span, on=['PatientId', 'PatternName', 'ComplianceType'], how='left')

# ------------------------------------------------------
# STEP 1b: Pattern-level aggregate (ComplianceType collapsed BEFORE the day
# collapse, in the correct order) - the single source of truth for anything
# Total-related, so this day-level step is never re-derived independently
# ------------------------------------------------------
daily_pattern = (
    daily.groupby(['PatientId', 'PatternName', 'Date'])['ComplianceScore']
    .mean()
    .reset_index()
)

df_agg_pattern = (
    daily_pattern.groupby(['PatientId', 'PatternName'])['ComplianceScore']
    .mean()
    .reset_index()
    .rename(columns={'ComplianceScore': 'AvgScore'})
)

pattern_min_start = (
    df.groupby(['PatientId', 'PatternName'])['StartDateTime']
    .min()
    .reset_index()
    .rename(columns={'StartDateTime': 'MinStartDateTime'})
)
df_agg_pattern = df_agg_pattern.merge(pattern_min_start, on=['PatientId', 'PatternName'], how='left')

print("Aggregation Complete. First 5 rows:")
print(df_agg.head())
print("\nPattern-level (for Total calculations):")
print(df_agg_pattern.head())
# Optional: Save this aggregated view to CSV
# df_agg.to_csv(RESULTS_DIR / 'aggregated_patient_scores.csv', index=False)

In [ ]:
# Attribute each patient's Total score to their admission time (min across their
# scored patterns) - AvgScore and MinStartDateTime already computed once, correctly
# (type collapsed before day collapse), in df_agg_pattern from Step 1b
patient_admission = df_agg_pattern.groupby('PatientId')['MinStartDateTime'].min().reset_index()

total_df = (
    df_agg_pattern.groupby('PatientId')['AvgScore']
    .mean()
    .reset_index()
    .rename(columns={'AvgScore': 'TotalAvgScore'})
    .merge(patient_admission, on='PatientId', how='left')
)


# --- Left panel data: Before/After split, using the properly-weighted Total ---
total_before_mask = (total_df["MinStartDateTime"] >= "2022-08-01") & (total_df["MinStartDateTime"] < "2023-08-01")
total_after_mask  = (total_df["MinStartDateTime"] >= "2024-08-01") & (total_df["MinStartDateTime"] < "2026-01-01")

before = total_df.loc[total_before_mask, 'TotalAvgScore']
after = total_df.loc[total_after_mask, 'TotalAvgScore']

avg_b, avg_a = before.mean(), after.mean()
if len(before) > 1 and len(after) > 1:
    stat, pval = stats.ttest_ind(before, after, equal_var=False)
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    p_text = f"p={pval:.4f}"
else:
    p_text, sig = "N/A", ""

base_color = "#000000"
before_color = to_rgba(base_color, alpha=0.45)
after_color = to_rgba(base_color, alpha=1.0)


# --- Right panel data: monthly mean + per-month 95% CI + one overall trend ---
total_plot = total_df[
    (total_df['MinStartDateTime'] >= '2022-08-01') & (total_df['MinStartDateTime'] < '2026-01-01')
].copy()
total_plot = total_plot.set_index('MinStartDateTime')

# Per-month sampling uncertainty (mean, SD, N -> 95% CI of the mean) - this is
# the patients-within-that-month uncertainty, not a regression-derived CI
monthly_stats = total_plot.resample('ME')['TotalAvgScore'].agg(['mean', 'std', 'count']).rename(
    columns={'mean': 'Mean', 'std': 'SD', 'count': 'N'}
)
monthly_stats['SEM'] = monthly_stats['SD'] / np.sqrt(monthly_stats['N'])
monthly_stats['CI_lower'] = monthly_stats['Mean'] - 1.96 * monthly_stats['SEM']
monthly_stats['CI_upper'] = monthly_stats['Mean'] + 1.96 * monthly_stats['SEM']

# One overall trend line across the whole visible time range, not segmented by period
trend_input = monthly_stats['Mean'].dropna()
x = np.arange(len(trend_input))
X = sm.add_constant(x)
trend_model = sm.OLS(trend_input.values, X).fit()
trend_fit = trend_model.predict(X)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
bars = ax1.bar(['Before', 'After'], [avg_b, avg_a], color=[before_color, after_color])
ax1.set_title(f"Total Adherence: Before vs After\n{sig} ({p_text})")
ax1.set_ylabel("Avg Score")
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
              f'{bar.get_height():.3f}', ha='center', va='bottom', fontweight='bold')

ax2 = axes[1]
ax2.plot(monthly_stats.index, monthly_stats['Mean'], color='gray', marker='o',
         linewidth=1.2, alpha=0.6, label='Monthly Avg')
ax2.fill_between(monthly_stats.index, monthly_stats['CI_lower'], monthly_stats['CI_upper'],
                  color='gray', alpha=0.2, label='95% CI (per month)')
ax2.plot(trend_input.index, trend_fit, color='#2a78d6', linestyle='--', linewidth=2, label='Overall trend')

ax2.axvline(pd.Timestamp("2023-08-01"), color="#1baf7a", linestyle="--", linewidth=1.5, label="Before cutoff")
ax2.axvline(pd.Timestamp("2024-08-01"), color="#eb6834", linestyle="--", linewidth=1.5, label="After cutoff")
ax2.set_xlim(monthly_stats.index.min(), pd.Timestamp("2025-12-31"))

ax2.set_title("Total Adherence Trend Over Time")
ax2.set_xlabel("Month")
ax2.set_ylabel("Avg Score")
ax2.grid(True)
ax2.legend(loc="best", fontsize=9)

fig.autofmt_xdate()

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------
# TABLE: Summary stats per Pattern + Compliance Type,
# bounded to the same window as df_final (no Before/After split)
# ------------------------------------------------------
df_agg_bounded = df_agg[
    (df_agg['MinStartDateTime'] >= '2022-08-01') &
    (df_agg['MinStartDateTime'] < '2026-01-01')
].copy()

summary = (
    df_agg_bounded.groupby(['PatternName', 'ComplianceType']).agg(
        N=('AvgScore', 'count'),
        Mean=('AvgScore', 'mean'),
        SD=('AvgScore', 'std'),
        Median=('AvgScore', 'median'),
        Q1=('AvgScore', lambda s: s.quantile(0.25)),
        Q3=('AvgScore', lambda s: s.quantile(0.75)),
    )
    .reset_index()
)

summary['IQR'] = summary['Q3'] - summary['Q1']
summary = summary.drop(columns=['Q1', 'Q3'])
summary = summary.rename(columns={'N': 'N relevant patients/admissions'})

summary

In [ ]:
# ------------------------------------------------------
# 2. TABLE (Grouped by Pattern AND Compliance Type)
# ------------------------------------------------------
from statsmodels.stats.weightstats import CompareMeans, DescrStatsW
from statsmodels.stats.multitest import multipletests

before_mask = (df_agg["MinStartDateTime"] >= "2022-08-01") & (df_agg["MinStartDateTime"] < "2023-08-01")
after_mask  = (df_agg["MinStartDateTime"] >= "2024-08-01") & (df_agg["MinStartDateTime"] < "2026-01-01")

df_agg_period = df_agg[before_mask | after_mask].copy()
df_agg_period["Period"] = np.where(before_mask[df_agg_period.index], "Before", "After")

# Aggregate to Patient level first to ensure N is correct
pat_agg = df_agg_pattern.groupby(['PatternName', 'ComplianceType', 'PatientId', 'Period'])['AvgScore'].mean().reset_index()

results = []
patterns = sorted(df_agg_pattern['PatternName'].unique())
c_types = sorted(df_agg_pattern['ComplianceType'].unique())

for pat in patterns:
    for c_type in c_types:
        # Filter for this specific combination
        d = pat_agg[(pat_agg['PatternName'] == pat) & (pat_agg['ComplianceType'] == c_type)]

        b_data = d[d['Period'] == 'Before']
        a_data = d[d['Period'] == 'After']

        b_scores = b_data['AvgScore']
        a_scores = a_data['AvgScore']

        # T-Test + 95% CI for the difference (Welch, matching equal_var=False above)
        if (
                len(b_scores) > 1
                and len(a_scores) > 1
                and b_scores.var() > 0
                and a_scores.var() > 0
            ):
            _, p = stats.ttest_ind(b_scores, a_scores, equal_var=False)
            cm = CompareMeans(DescrStatsW(a_scores), DescrStatsW(b_scores))
            ci_low, ci_high = cm.tconfint_diff(usevar='unequal')
        else:
            p = np.nan
            ci_low, ci_high = np.nan, np.nan

        results.append({
            'PatternName': pat,
            'ComplianceType': c_type,
            'Before Avg': round(b_scores.mean(), 2),
            'After Avg': round(a_scores.mean(), 2),
            'Difference': round(a_scores.mean() - b_scores.mean(), 2),
            'CI Lower': round(ci_low, 3) if pd.notna(ci_low) else np.nan,
            'CI Upper': round(ci_high, 3) if pd.notna(ci_high) else np.nan,
            'P-Value': round(p, 5),
            'N Before': b_data['PatientId'].nunique(),
            'N After': a_data['PatientId'].nunique()
        })

res = pd.DataFrame(results)
res = res[(res['N Before'] > 0) | (res['N After'] > 0)]

# BH-FDR correction across the component-level comparisons that were actually testable
valid = res['P-Value'].notna()
if valid.sum() > 0:
    rejected, p_adj, _, _ = multipletests(res.loc[valid, 'P-Value'], method='fdr_bh')
    res.loc[valid, 'P-Value (FDR)'] = p_adj
    res.loc[valid, 'Significant (FDR)'] = rejected
else:
    res['P-Value (FDR)'] = np.nan
    res['Significant (FDR)'] = False

res

In [ ]:
# ------------------------------------------------------
# 3. PLOT (Adherance Before vs After, by Hospital)
# ------------------------------------------------------

# 1. Load your files (same pattern as mortality analysis)
df = df_agg_bounded.copy()
visit_master = pd.read_csv(EXTERNAL_DATA_DIR / 'visits_master.csv')

# 2. Merge Data
# Inner join on PatientId = visit_id
df_merged = df.merge(visit_master, left_on='PatientId', right_on='visit_id', how='inner')

# 3. Filter Date (Aug 2022 Onward) + duration cap, same as the mortality cell
df_final = df_merged[
    (df_merged['MinStartDateTime'] >= '2022-08-01') & (df_merged['MinStartDateTime'] < '2026-01-01')
].copy()
df_final['duration_days_int'] = df_final['duration_days'].round().astype(int)
df_final = df_final[df_final['duration_days_int'] <= 14]

# 4. Assign Before/After period (same cutoffs used throughout the notebook, gap excluded)
before_mask = (df_final['MinStartDateTime'] >= '2022-08-01') & (df_final['MinStartDateTime'] < '2023-08-01')
after_mask  = (df_final['MinStartDateTime'] >= '2024-08-01') & (df_final['MinStartDateTime'] < '2026-01-01')
df_final = df_final[before_mask | after_mask].copy()
df_final['Period'] = np.where(before_mask[df_final.index], 'Before', 'After')

# 5. Patient-level average per hospital + period (one score per patient first)
patient_avg = df_final.groupby(['PatientId', 'hospital', 'Period'])['AvgScore'].mean().reset_index()

hospitals = sorted(patient_avg['hospital'].dropna().unique())

# 6. Per-hospital Before/After comparison
from statsmodels.stats.multitest import multipletests

results = []
for hosp in hospitals:
    d = patient_avg[patient_avg['hospital'] == hosp]
    before = d[d['Period'] == 'Before']['AvgScore']
    after = d[d['Period'] == 'After']['AvgScore']

    if len(before) > 1 and len(after) > 1 and before.var() > 0 and after.var() > 0:
        _, p = stats.ttest_ind(before, after, equal_var=False)
    else:
        p = np.nan

    results.append({
        'hospital': hosp,
        'Before Avg': before.mean(),
        'After Avg': after.mean(),
        'N Before': len(before),
        'N After': len(after),
        'P-Value': p,
    })

hosp_res = pd.DataFrame(results)

# BH-FDR correction across hospitals
valid = hosp_res['P-Value'].notna()
if valid.sum() > 0:
    rejected, p_adj, _, _ = multipletests(hosp_res.loc[valid, 'P-Value'], method='fdr_bh')
    hosp_res.loc[valid, 'P-Value (FDR)'] = p_adj
    hosp_res.loc[valid, 'Significant (FDR)'] = rejected
else:
    hosp_res['P-Value (FDR)'] = np.nan
    hosp_res['Significant (FDR)'] = False

# 7. Plot: grouped bars, one group per hospital, Before/After side by side
fig, ax = plt.subplots(figsize=(max(8, len(hospitals) * 1.5), 6))

x = np.arange(len(hospitals))
width = 0.35

before_vals = hosp_res.set_index('hospital').loc[hospitals, 'Before Avg']
after_vals = hosp_res.set_index('hospital').loc[hospitals, 'After Avg']

before_color = to_rgba('#2a78d6', alpha=0.5)
after_color = to_rgba('#2a78d6', alpha=1.0)

ax.bar(x - width / 2, before_vals, width, label='Before', color=before_color)
ax.bar(x + width / 2, after_vals, width, label='After', color=after_color)

ax.set_xticks(x)
ax.set_xticklabels(hospitals, rotation=30, ha='right')
ax.set_ylabel('Avg Compliance Score')
ax.set_title('Adherence Before vs After Intervention, by Hospital')
ax.legend()

# Significance marker per hospital (FDR-corrected)
for i, hosp in enumerate(hospitals):
    row = hosp_res[hosp_res['hospital'] == hosp].iloc[0]
    p_fdr = row['P-Value (FDR)']
    if pd.notna(p_fdr):
        sig = '***' if p_fdr < 0.001 else '**' if p_fdr < 0.01 else '*' if p_fdr < 0.05 else 'ns'
    else:
        sig = 'N/A'
    y_max = max(row['Before Avg'], row['After Avg'])
    ax.text(i, y_max + 0.01, sig, ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

hosp_res


In [ ]:
# ------------------------------------------------------
# 3. PLOT (Correlation between adherance to Mortality)
# ------------------------------------------------------

# 1. Load your files (pattern-level, equal-weighted aggregate - not per-ComplianceType)
df_agg_pattern_bounded = df_agg_pattern[
    (df_agg_pattern['MinStartDateTime'] >= '2022-08-01') & (df_agg_pattern['MinStartDateTime'] < '2026-01-01')
].copy()

df = df_agg_pattern_bounded.copy()
visit_master = pd.read_csv(EXTERNAL_DATA_DIR / 'visits_master.csv')

# 2. Merge Data
# Inner join on PatientId = visit_id
df_merged = df.merge(visit_master, left_on='PatientId', right_on='visit_id', how='inner')

# 3. Filter Date (Aug 2022 Onward)
df_final = df_merged[(df_merged['MinStartDateTime'] >= '2022-08-01')&(df_merged['MinStartDateTime'] < '2026-01-01')].copy()
df_final['duration_days_int'] = df_final['duration_days'].round().astype(int)
df_final = df_final[df_final['duration_days_int'] <= 14]

# 4. Analyze: Group by Patient first to get 1 score per patient - AvgScore is
#    already ComplianceType-collapsed per (PatientId, PatternName), so this now
#    correctly collapses across PATTERNS equally, not across raw (pattern, type) rows
patient_avg = df_final.groupby(['PatientId', 'death_within_30d'])['AvgScore'].mean().reset_index()
duration_avg = (
    df_final
    .groupby('duration_days_int')['AvgScore']
    .mean()
    .reset_index()
    .sort_values('duration_days_int')
)

# Separate groups
alive = patient_avg[patient_avg['death_within_30d'] == False]['AvgScore']
deceased = patient_avg[patient_avg['death_within_30d'] == True]['AvgScore']

# Calculate Stats
mean_alive = alive.mean()
mean_deceased = deceased.mean()

# Build compliance-score bins and compute mortality ratio per bin
n_bins = 10

patient_avg_plot = patient_avg.dropna(subset=['AvgScore', 'death_within_30d']).copy()

# Create equal-width bins over AvgScore
patient_avg_plot['score_bin'] = pd.cut(
    patient_avg_plot['AvgScore'],
    bins=n_bins,
    include_lowest=True
)

bin_stats = (
    patient_avg_plot
    .groupby('score_bin', observed=False)
    .agg(
        n_patients=('death_within_30d', 'size'),
        mortality_ratio=('death_within_30d', 'mean'),
        avg_score=('AvgScore', 'mean')
    )
    .reset_index()
)

# optional: remove very sparse bins if needed
bin_stats = bin_stats[bin_stats['n_patients'] >= 30].copy()

# T-Test (left plot): do the two groups differ on mean adherence?
if len(alive) > 1 and len(deceased) > 1:
    t_stat, p_val = stats.ttest_ind(alive, deceased, equal_var=False)
    p_text = f"p={p_val:.4f}"
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
else:
    p_text, sig = "N/A", ""

# ------------------------------------------------------
# Trend test (right plot): "the higher we go in adherence,
# the lower we go in mortality" -- a dose-response question,
# which needs a TREND test, not a two-group t-test.
#
# Logistic regression on the CONTINUOUS score is the most powerful choice:
# it loses no information to binning and uses every patient's sample weight.
# It tests H0: adherence has no effect on the log-odds of death. A negative
# slope (OR < 1) means higher adherence is associated with lower mortality.
# ------------------------------------------------------
lr_df = patient_avg_plot[['AvgScore', 'death_within_30d']].dropna().copy()
X_lr = sm.add_constant(lr_df['AvgScore'])                 # intercept + slope
y_lr = lr_df['death_within_30d'].astype(int)
logit_res = sm.Logit(y_lr, X_lr).fit(disp=0)

beta = logit_res.params['AvgScore']                        # log-odds change per +1.0 score
or_per_unit = np.exp(beta)                                 # odds ratio per +1.0 score
or_per_0_1 = np.exp(beta * 0.1)                            # odds ratio per +0.1 score (digestible)
ci_low, ci_high = np.exp(logit_res.conf_int().loc['AvgScore'] * 0.1)  # 95% CI on OR per +0.1
p_trend = logit_res.pvalues['AvgScore']
trend_sig = "***" if p_trend < 0.001 else "**" if p_trend < 0.01 else "*" if p_trend < 0.05 else "ns"

print("=== Right-plot trend test: does mortality fall as adherence rises? ===")
print(f"Logistic regression (continuous score):  "
      f"OR={or_per_0_1:.3f} per +0.1 adherence  (95% CI {ci_low:.3f}-{ci_high:.3f}), "
      f"p={p_trend:.4g}  {trend_sig}")
print(f"Direction: {'higher adherence -> LOWER mortality' if beta < 0 else 'higher adherence -> higher mortality'}")

# 5. Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --------------------
# Plot 1: Mortality bars
# --------------------
ax1 = axes[0]

left_colors = ['#9ecae1', '#3182bd']  # light blue, medium-dark blue

bars1 = ax1.bar(
    ['Alive (False)', 'Deceased (True)'],
    [mean_alive, mean_deceased],
    color=left_colors
)

ax1.set_title(f"Avg Compliance Score by Mortality Status\n{sig} ({p_text})")
ax1.set_ylabel("Avg Compliance Score")
ax1.set_xlabel("Death Within 30 Days")
# tilt x tick labels
plt.setp(ax1.get_xticklabels(), rotation=30, ha='right')

# annotate bars
left_label_offset = max(mean_alive, mean_deceased) * 0.015
for bar in bars1:
    h = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        h + left_label_offset,
        f"{h:.3f}",
        ha='center',
        va='bottom',
        fontweight='bold'
    )

# adapt frame height to bar + text
ax1.set_ylim(0, max(mean_alive, mean_deceased) + left_label_offset * 4)

# --------------------
# Plot 2: Mortality Ratio by Compliance Score Bin
# --------------------
ax2 = axes[1]

# dark-to-light blue palette from low compliance to high
right_colors = plt.cm.Blues(np.linspace(0.85, 0.35, len(bin_stats)))

bars2 = ax2.bar(
    range(len(bin_stats)),
    bin_stats['mortality_ratio'],
    width=0.9,
    color=right_colors
)

# readable x labels
bin_labels = [
    f"{interval.left:.2f}-{interval.right:.2f}"
    for interval in bin_stats['score_bin']
]

ax2.set_xticks(range(len(bin_stats)))
ax2.set_xticklabels(bin_labels, rotation=45, ha='right')

# Headline the logistic-regression trend result (OR per +0.1 score is the
# most digestible effect size when AvgScore lives on a 0-1 scale).
ax2.set_title(
    "Mortality Ratio by Compliance Score Bin\n"
    f"trend {trend_sig} (OR={or_per_0_1:.2f} per +0.1 adherence, p={p_trend:.4f})"
)
ax2.set_xlabel("Compliance Score Bin")
ax2.set_ylabel("Mortality Ratio")
ax2.margins(x=0.01)

# annotate bars
right_label_offset = bin_stats['mortality_ratio'].max() * 0.03
for bar, (_, row) in zip(bars2, bin_stats.iterrows()):
    height = bar.get_height()

    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        height * 0.55,  # middle of the bar
        f"{height:.2f}\n(n={int(row['n_patients'])})",
        ha='center',
        va='center',
        fontsize=9,
        color='white',
        fontweight='bold'
    )

# adapt frame height to bar + text
ax2.set_ylim(0, bin_stats['mortality_ratio'].max() + right_label_offset * 5)

# # smoothed trend line over bin centers
# x = np.arange(len(bin_stats))
# y = bin_stats['mortality_ratio'].values

# if len(x) >= 4:
#     x_smooth = np.linspace(x.min(), x.max(), 300)
#     spline = make_interp_spline(x, y, k=3)
#     y_smooth = spline(x_smooth)
#     ax2.plot(x_smooth, y_smooth, color='#08306b', linewidth=2.5, alpha=0.9)

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------
# 4. ADJUSTED ANALYSIS
# Does the AvgScore-mortality association persist after adjusting for
# age, sex, hospital, and baseline clinical status (comorbidity burden)?
# ------------------------------------------------------
import statsmodels.api as sm

# Reuse the cleaned static feature set (age, gender, comorbidities, CCI score)
static_clean = pd.read_csv(RESULTS_DIR / "context_static_input_clean.csv")

adj_df = patient_avg.merge(
    df_final[['PatientId', 'hospital', 'person_id']].drop_duplicates(),
    on='PatientId', how='left'
)
adj_df = adj_df.merge(
    static_clean.drop(columns=['person_id'], errors='ignore'),
    left_on='PatientId', right_on='VisitId', how='left'
)

# Prespecified covariates: sex, age, baseline severity; hospital as fixed effect (dummies)
covariates = ['age_years', 'gender_code', 'CCI_LIKE_SCORE']
adj_df = pd.get_dummies(adj_df, columns=['hospital'], prefix='hosp', drop_first=True)
hosp_dummy_cols = [c for c in adj_df.columns if c.startswith('hosp_')]

feature_cols = ['AvgScore'] + covariates + hosp_dummy_cols
adj_df = adj_df.dropna(subset=feature_cols + ['death_within_30d', 'person_id'])

X = adj_df[feature_cols].apply(pd.to_numeric, errors='coerce')
X = (X - X.mean()) / X.std(ddof=0).replace(0, 1)  # standardize for comparable coefficients
X = sm.add_constant(X, has_constant='add')
y = adj_df['death_within_30d'].astype(int)

# Cluster-robust SEs on person_id (not PatientId/admission ID - adj_df is already
# one row per admission, so clustering on PatientId would be singleton clusters
# and correct for nothing; person_id is the real repeated-measures unit, since
# the same physical patient can have multiple admissions in this cohort)
model = sm.Logit(y, X).fit(
    disp=0,
    cov_type='cluster',
    cov_kwds={'groups': adj_df['person_id']}
)
print(model.summary())

# Focused readout: does the AvgScore association survive adjustment?
coef = model.params['AvgScore']
p = model.pvalues['AvgScore']
ci = model.conf_int().loc['AvgScore']
print(f"\nAvgScore (adjusted): coef={coef:.3f}, OR={np.exp(coef):.3f}, "
      f"95% CI=({np.exp(ci[0]):.3f}, {np.exp(ci[1]):.3f}), p={p:.4f}")


In [ ]:
# ------------------------------------------------------
# ADHERENCE BY DIABETES-DIAGNOSIS STATUS
# (restricted to patients who were HYPERGLYCEMIC within the first 48h)
#
# Mirror of the LEFT bar of the mortality analysis, but comparing patients WITH
# vs WITHOUT a documented diabetes diagnosis - AND restricting BOTH groups to
# patients who actually went hyperglycemic in the first 48h, so diagnosed
# patients who never had hyperglycemia are excluded. One adherence score per
# patient (equal-weighted across patterns), Welch t-test between the two groups.
# ------------------------------------------------------

# The documented diabetes-diagnosis flag lives in the cleaned STATIC table
# (context_static_input_clean.csv), the same source the adjusted mortality model
# uses for background columns (age, gender, CCI) - NOT in visit_master.
DX_COL = "DIABETES_DIAGNOSIS"

# First-48h hyperglycemia restriction (set False to reproduce the whole-cohort bar)
RESTRICT_TO_HYPER_48H = True
EARLY_WINDOW_HOURS = 48
HYPER_CONCEPTS = ["HYPERGLYCEMIA_EVENT", "SEVERE_HYPERGLYCEMIA_EVENT"]  # "hyperglycemic"

# 1. Same data prep / bounds as the mortality left-bar plot (cell "3. PLOT ...")
df_dx = df_agg_pattern[
    (df_agg_pattern['MinStartDateTime'] >= '2022-08-01') & (df_agg_pattern['MinStartDateTime'] < '2026-01-01')
].copy()
visit_master = pd.read_csv(EXTERNAL_DATA_DIR / 'visits_master.csv')
merged = df_dx.merge(visit_master, left_on='PatientId', right_on='visit_id', how='inner')
merged = merged[
    (merged['MinStartDateTime'] >= '2022-08-01') & (merged['MinStartDateTime'] < '2026-01-01')
].copy()
merged['duration_days_int'] = merged['duration_days'].round().astype(int)
merged = merged[merged['duration_days_int'] <= 14]

# 2. Bring in the diabetes-diagnosis flag from the cleaned static table, keyed by
#    VisitId (same merge the adjusted model uses to pull age/CCI). It is NOT in
#    visit_master, so this join is what makes the flag available.
static_clean = pd.read_csv(RESULTS_DIR / "context_static_input_clean.csv")
if DX_COL not in static_clean.columns:
    raise KeyError(
        f"'{DX_COL}' not found in context_static_input_clean.csv. "
        f"Diabetes-ish columns available there: {[c for c in static_clean.columns if 'diab' in c.lower()]}"
    )
merged = merged.merge(
    static_clean[['VisitId', DX_COL]],
    left_on='PatientId', right_on='VisitId', how='left'
)

# Keep only patients with a recorded diagnosis status (drop unmatched/missing)
n_before = merged['PatientId'].nunique()
merged = merged.dropna(subset=[DX_COL])
if merged['PatientId'].nunique() < n_before:
    print(f"Dropped {n_before - merged['PatientId'].nunique()} patients with no diabetes-diagnosis flag.")

# 3. Restrict to patients who were hyperglycemic within the first 48h. Reuses the
#    same admission proxy (earliest scored timestamp) and 48h early window as the
#    glycemic-event analysis; the flag is built from the raw event stream.
if RESTRICT_TO_HYPER_48H:
    # Admission proxy per patient = earliest scored timestamp (== qa_df min; equals
    # the admission_map used in the glycemic-event cell)
    admission_map = pd.to_datetime(df_agg_pattern.groupby('PatientId')['MinStartDateTime'].min())
    if admission_map.dt.tz is not None:
        admission_map = admission_map.dt.tz_localize(None)

    events_df = read_output_data(parse_dates=['StartDateTime'])
    hyper_ev = events_df[events_df['ConceptName'].isin(HYPER_CONCEPTS)].copy()
    ev_time = pd.to_datetime(hyper_ev['StartDateTime'])
    if ev_time.dt.tz is not None:
        ev_time = ev_time.dt.tz_localize(None)
    hyper_ev['_evt'] = ev_time

    # Event within [admission, admission + 48h]
    cutoff = hyper_ev['PatientId'].map(admission_map) + pd.Timedelta(hours=EARLY_WINDOW_HOURS)
    hyper_48h_patients = set(
        hyper_ev.loc[hyper_ev['_evt'] <= cutoff, 'PatientId'].unique()
    )

    n_pre = merged['PatientId'].nunique()
    merged = merged[merged['PatientId'].isin(hyper_48h_patients)].copy()
    print(f"First-48h hyperglycemia restriction: kept {merged['PatientId'].nunique()} "
          f"of {n_pre} patients ({len(hyper_48h_patients)} hyperglycemic-in-48h overall).")

# 4. Normalize the flag to a boolean 'has diagnosis' (handles bool / 0-1 / text)
raw = merged[DX_COL]
if raw.dtype == bool:
    merged['has_dx'] = raw
elif pd.api.types.is_numeric_dtype(raw):
    merged['has_dx'] = raw > 0
else:
    merged['has_dx'] = raw.astype(str).str.strip().str.lower().isin(['1', 'y', 'yes', 'true', 't'])

print("Unique patients per group:")
print(merged.groupby('has_dx')['PatientId'].nunique())

# 5. Collapse to one adherence score per patient (equal-weighted across patterns),
#    then split by diagnosis status
patient_dx = merged.groupby(['PatientId', 'has_dx'])['AvgScore'].mean().reset_index()
with_dx = patient_dx[patient_dx['has_dx']]['AvgScore']
without_dx = patient_dx[~patient_dx['has_dx']]['AvgScore']
mean_with, mean_without = with_dx.mean(), without_dx.mean()

# 6. Welch t-test (unequal variance), same as the mortality bars
if len(with_dx) > 1 and len(without_dx) > 1:
    t_stat, p_val = stats.ttest_ind(with_dx, without_dx, equal_var=False)
    p_text = f"p={p_val:.4f}"
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
else:
    p_text, sig = "N/A", ""

print(f"\nAvg adherence | with dx: {mean_with:.3f} (n={len(with_dx)})  |  "
      f"without dx: {mean_without:.3f} (n={len(without_dx)})  |  {sig} ({p_text})")
print(f"Direction: patients {'WITH' if mean_with > mean_without else 'WITHOUT'} a diagnosis "
      f"received better-adhered care")

# 7. Plot - left-bar style, matching the mortality figure's left panel
cohort_note = "hyperglycemic within 48h" if RESTRICT_TO_HYPER_48H else "all eligible admissions"
fig, ax = plt.subplots(figsize=(6, 6))
colors = ['#9ecae1', '#3182bd']  # no-diagnosis light blue, diagnosis dark blue

bars = ax.bar(
    ['No diagnosis', 'Diabetes diagnosis'],
    [mean_without, mean_with],
    color=colors
)

ax.set_title(f"Avg Compliance Score by Diabetes Diagnosis\n({cohort_note})   {sig} ({p_text})")
ax.set_ylabel("Avg Compliance Score")
ax.set_xlabel("Documented Diabetes Diagnosis")
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

off = max(mean_with, mean_without) * 0.015
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + off, f"{h:.3f}",
            ha='center', va='bottom', fontweight='bold')

ax.set_ylim(0, max(mean_with, mean_without) + off * 4)
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------
# specific format to ensure dates are read correctly
events_df = read_output_data(parse_dates=['StartDateTime', 'EndDateTime'])
qa_df = read_qa_scores(parse_dates=['StartDateTime', 'EndDateTime'])

In [ ]:
# ============================================================
# Adherence vs Glycemic Events (Daily)
# ============================================================
INSULIN_PATTERNS = [
    'ROUTINE_BASAL_DOSAGE_PATTERN',
    'CONTINUE_BASAL_ON_ADMISSION_PATTERN',
    'INSULIN_ON_HYPERGLYCEMIA_PATTERN',
    # 'REDUCE_BASAL_ON_HYPOGLYCEMIA_PATTERN',
    # 'REDUCE_BOLUS_ON_HYPOGLYCEMIA_PATTERN',
    # 'STOP_ANTIDIABETICS_ON_ADMISSION_PATTERN',
    # 'STOP_ANTIDIABETICS_ON_HYPOGLYCEMIA_PATTERN',
    # 'STOP_SGLT2_ON_ADMISSION_PATTERN',
    # 'CONTINUE_METFORMIN_ON_ADMISSION_PATTERN',
]

EVENT_CONCEPTS = ["HYPERGLYCEMIA_EVENT", "HYPOGLYCEMIA_EVENT",
                   "SEVERE_HYPERGLYCEMIA_EVENT", "SEVERE_HYPOGLYCEMIA_EVENT"]

# ============================================================
# Adherence vs Glycemic Events (Patient-Level Correlation)
# ============================================================
USE_EARLY_WINDOW_ONLY = False  # if True: adherence from first 48h only, event ratio from day 3 onward
EARLY_WINDOW_HOURS = 48

insulin_qa = qa_df[qa_df['PatternName'].isin(INSULIN_PATTERNS)].copy()
insulin_qa['StartDateTime'] = pd.to_datetime(insulin_qa['StartDateTime'])
insulin_qa['Date'] = insulin_qa['StartDateTime'].dt.date

# Admission-time proxy: each patient's earliest QA-scored timestamp.
# Swap for an actual ADMISSION event timestamp if you have one loaded — more precise.
qa_df['StartDateTime'] = pd.to_datetime(qa_df['StartDateTime'])
admission_map = qa_df.groupby('PatientId')['StartDateTime'].min()

if USE_EARLY_WINDOW_ONLY:
    cutoff = insulin_qa['PatientId'].map(admission_map) + pd.Timedelta(hours=EARLY_WINDOW_HOURS)
    insulin_qa = insulin_qa[insulin_qa['StartDateTime'] <= cutoff]  # days 1-2

# 1. Patient-level insulin-adherence score, same day-then-type-then-pattern-then-
#    patient equal-weighting hierarchy used for the Total adherence score elsewhere:
#    (patient, pattern, complianceType, day) -> (patient, pattern, day)
#    -> (patient, pattern) -> (patient)
daily_type = (
    insulin_qa
    .groupby(['PatientId', 'PatternName', 'ComplianceType', 'Date'])['ComplianceScore']
    .mean()
    .reset_index()
)
daily_pattern = (
    daily_type
    .groupby(['PatientId', 'PatternName', 'Date'])['ComplianceScore']
    .mean()
    .reset_index()
)
pattern_avg = (
    daily_pattern
    .groupby(['PatientId', 'PatternName'])['ComplianceScore']
    .mean()
    .reset_index()
)
patient_adherence = (
    pattern_avg
    .groupby('PatientId')['ComplianceScore']
    .mean()
    .reset_index()
    .rename(columns={'ComplianceScore': 'PatientScore'})
)

# 2. Stay length per patient (reuse the same filtered duration used in the mortality analysis)
patient_days = df_final[['PatientId', 'duration_days_int']].drop_duplicates()

# 3. Event-day counts per patient from OutputPatientData
events_df = events_df[events_df['ConceptName'].isin(EVENT_CONCEPTS)].copy()
events_df['StartDateTime'] = pd.to_datetime(events_df['StartDateTime'])
events_df['Date'] = events_df['StartDateTime'].dt.date

if USE_EARLY_WINDOW_ONLY:
    cutoff = events_df['PatientId'].map(admission_map) + pd.Timedelta(hours=EARLY_WINDOW_HOURS)
    events_df = events_df[events_df['StartDateTime'] > cutoff]  # day 3 onward — strictly after the adherence window

daily_events = (
    events_df
    .groupby(['PatientId', 'Date', 'ConceptName'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in EVENT_CONCEPTS:
    if col not in daily_events.columns:
        daily_events[col] = 0

daily_events['has_hyper_day'] = daily_events['HYPERGLYCEMIA_EVENT'] > 0
daily_events['has_severe_hyper_day'] = daily_events['SEVERE_HYPERGLYCEMIA_EVENT'] > 0
daily_events['has_hypo_day'] = daily_events['HYPOGLYCEMIA_EVENT'] > 0
daily_events['has_severe_hypo_day'] = daily_events['SEVERE_HYPOGLYCEMIA_EVENT'] > 0

FLAG_COLS = ['has_hyper_day', 'has_severe_hyper_day', 'has_hypo_day', 'has_severe_hypo_day']
event_day_counts = daily_events.groupby('PatientId')[FLAG_COLS].sum().reset_index()

# 4. Assemble patient-level table: score + event-day ratios
patient_stats = patient_adherence.merge(patient_days, on='PatientId', how='inner')
patient_stats = patient_stats.merge(event_day_counts, on='PatientId', how='left')
patient_stats[FLAG_COLS] = patient_stats[FLAG_COLS].fillna(0)

if USE_EARLY_WINDOW_ONLY:
    # A patient needs a real day-3-onward period to contribute a meaningful ratio;
    # otherwise the ratio is fabricated (0/0 forced to 0/1 by the clip below)
    patient_stats = patient_stats[patient_stats['duration_days_int'] > EARLY_WINDOW_HOURS / 24].copy()

# Denominator: remaining observed days after the early window, if the flag is on
denom = patient_stats['duration_days_int'] - (EARLY_WINDOW_HOURS / 24 if USE_EARLY_WINDOW_ONLY else 0)
denom = denom.clip(lower=1)

for col in FLAG_COLS:
    ratio_col = col.replace('has_', '').replace('_day', '_ratio')
    patient_stats[ratio_col] = (patient_stats[col] / denom).clip(upper=1.0)


def plot_adherence_correlation(patient_df, ratio_col_regular, ratio_col_severe,
                                label_regular, label_severe, color):
    """
    Purpose: Correlate patient-level insulin-adherence score against the proportion
             of stay-days with a glycemic event, for a regular event and its severe counterpart.
    Method: Scatter + least-squares regression line per subplot, annotated with
            Pearson r, its p-value, and a Cohen's-r effect-size label.

    Args:
        patient_df (pd.DataFrame): One row per patient with PatientScore and event ratios.
        ratio_col_regular (str): Column name for the regular event's day-ratio.
        ratio_col_severe (str): Column name for the severe event's day-ratio.
        label_regular (str): Display label for the regular event subplot.
        label_severe (str): Display label for the severe event subplot.
        color (str): Scatter/line color for both subplots.

    Returns:
        None: Displays the figure.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    for ax, ratio_col, label in zip(axes, [ratio_col_regular, ratio_col_severe], [label_regular, label_severe]):
        x = patient_df['PatientScore']
        y = patient_df[ratio_col]

        ax.scatter(x, y, color=color, alpha=0.5, edgecolors='none')

        if len(x) > 2 and x.var() > 0 and y.var() > 0:
            r, p_val = stats.pearsonr(x, y)
            slope, intercept, *_ = stats.linregress(x, y)
            x_line = np.linspace(x.min(), x.max(), 100)
            ax.plot(x_line, slope * x_line + intercept, color='black', linewidth=2.5, zorder=5)

            abs_r = abs(r)
            effect = "negligible" if abs_r < 0.1 else "small" if abs_r < 0.3 else "medium" if abs_r < 0.5 else "large"
            title = f"{label}\nr={r:.3f} (n={len(x)}), p={p_val:.4f}, effect={effect}"
        else:
            title = f"{label}\nNot enough variation to compute correlation"

        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Patient Avg Insulin-Adherence Score")
        ax.set_ylabel(f"{label} Day Ratio")

    plt.tight_layout()
    plt.show()


# Question 1: Hyperglycemia vs Severe Hyperglycemia
plot_adherence_correlation(
    patient_stats, 'hyper_ratio', 'severe_hyper_ratio',
    'Hyperglycemia', 'Severe Hyperglycemia', color='#c98500'
)

# Question 2: Hypoglycemia vs Severe Hypoglycemia
plot_adherence_correlation(
    patient_stats, 'hypo_ratio', 'severe_hypo_ratio',
    'Hypoglycemia', 'Severe Hypoglycemia', color='#3182bd'
)

# 🔍 Finding Importance in the Abstracted Data

**Data:** One row per (visit, complication) pair, labeled `1` if the complication first occurred at/after a cutoff (tested at 24h/48h/72h from admission) or `0` if it never occurred. Visits where the complication occurred *before* the cutoff are excluded (can't attribute a pre-existing condition to in-stay care). Adherence (QA) and baseline patient (STATIC) features are computed only from admission up to that cutoff, never after.

**Method:** One Firth (bias-reduced) logistic regression per complication per cutoff, with QA and STATIC features together — so each adherence coefficient is "holding measured comorbidity/severity constant." Gives real p-values. A lightweight bootstrap sign-agreement check flags fragile estimates. A feature is only called **robust** (starred) if significant in **all three** cutoff variants, not just one.

**Limitations:** Association, not proven causation — adjusting for comorbidities reduces but doesn't eliminate confounding by illness severity. Correlated features aren't fully separable, so a hit may reflect a cluster of related patterns, not one isolated driver. One known circularity (`ROUTINE_GLUCOSE_MEASURE_PATTERN` vs. hyper/hypo events) was found and excluded — others may exist unaudited. Coefficients are standardized, not calibrated real-world effect sizes.


In [ ]:
import pandas as pd
import numpy as np
from typing import Optional, List

admission_event = 'ADMISSION_EVENT'
release_event = 'RELEASE_EVENT'
complication_concepts = {'DEATH_EVENT',
                          "HYPERGLYCEMIA_EVENT",
                          "HYPOGLYCEMIA_EVENT",
                          "SEVERE_HYPERGLYCEMIA_EVENT",
                          "SEVERE_HYPOGLYCEMIA_EVENT",
                          "KIDNEY_COMPLICATION_EVENT",
                          "INFECTION_EVENT",
                          "CARDIO-VASCULAR_DISORDER_EVENT",
                          "HYPEROSMOLALITY_EVENT",
                          "KETOACIDOSIS_EVENT"
                          }


def validate_visit_id_coverage(visit_master, output_df, qa_df, static_df):
    """
    Validate VisitId consistency across all source tables.

    Reports:
    - number of unique VisitIds per table
    - IDs missing from each table relative to the others
    - IDs appearing only in one table
    """

    tables = {
        "visit_master": set(visit_master["VisitId"].dropna().unique()),
        "output_df": set(output_df["VisitId"].dropna().unique()),
        "qa_df": set(qa_df["VisitId"].dropna().unique()),
        "static_df": set(static_df["VisitId"].dropna().unique()),
    }

    print("---- Unique VisitId counts ----")
    for name, ids in tables.items():
        print(f"{name:12}: {len(ids):>8}")

    all_ids = set.union(*tables.values())

    print("\n---- Missing IDs per table ----")
    for name, ids in tables.items():
        missing = all_ids - ids
        print(f"{name:12}: missing {len(missing)}")

    print("\n---- Pairwise mismatches ----")
    names = list(tables.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            diff_ab = tables[a] - tables[b]
            diff_ba = tables[b] - tables[a]
            print(f"{a} not in {b}: {len(diff_ab)}")
            print(f"{b} not in {a}: {len(diff_ba)}")

    print("\n---- IDs appearing in only one table ----")
    for name, ids in tables.items():
        others = set.union(*(tables[n] for n in tables if n != name))
        only_here = ids - others
        print(f"{name:12}: {len(only_here)}")


def column_summary(df):
    summary = []

    for col in df.columns:
        s = df[col]

        summary.append({
            "column": col,
            "dtype": str(s.dtype),
            "null_count": s.isna().sum(),
            "null_pct": round(s.isna().mean(), 4),
            "unique_values": s.nunique(),
            "min": s.min() if pd.api.types.is_numeric_dtype(s) else None,
            "max": s.max() if pd.api.types.is_numeric_dtype(s) else None,
            "mean": s.mean() if pd.api.types.is_numeric_dtype(s) else None,
            "median": s.median() if pd.api.types.is_numeric_dtype(s) else None,
        })

    return pd.DataFrame(summary)


visit_master = pd.read_csv(EXTERNAL_DATA_DIR / 'visits_master.csv')
output_df = read_output_data(low_memory=False)
qa_df = read_qa_scores(low_memory=False)
static_df = pd.read_csv(EXTERNAL_DATA_DIR / "context_static_input.csv", low_memory=False)

# Standardize visit id column name
# You said PatientId is actually visit id in these DFs
output_df = output_df.rename(columns={"PatientId": "VisitId"})
qa_df = qa_df.rename(columns={"PatientId": "VisitId"})
static_df = static_df.rename(columns={"visit_id": "VisitId"})

# visit_master uses visit_id
# keep only what you need
# drop duplicated visit_ids by removing the whole row
visit_master = visit_master.rename(columns={"visit_id": "VisitId",
                                             "start_datetime": "StartDateTime",
                                             "end_datetime": "EndDateTime"})

# count how many unique persons per VisitId
visit_person_counts = (
    visit_master.groupby("VisitId")["person_id"]
    .nunique()
)

# VisitIds that are linked to more than one person are removed, we keep only IDs that are in Static
invalid_visit_ids = visit_person_counts[visit_person_counts > 1].index
relevant_ids_subset = static_df["VisitId"].unique()
visit_master = visit_master[~visit_master["VisitId"].isin(invalid_visit_ids)].copy()
visit_master = visit_master[visit_master["VisitId"].isin(relevant_ids_subset)].copy()
visit_master = (
    visit_master.sort_values("StartDateTime")
    .drop_duplicates(subset=["VisitId", "person_id"], keep="last")
)

# Quick validation (after reduce) on purpose
validate_visit_id_coverage(
    visit_master,
    output_df,
    qa_df,
    static_df
)


In [ ]:
# ============================================================
# Static Features: Causality-Phase Cleanup
# (assumes static_df already went through Data-Driven Quality Cleaning above -
#  hardcoded range clipping, IQR clipping, zero-variance drop, redundant-numeric drop)
# ============================================================
STATIC_ID_COLS = ['VisitId', 'person_id']
MISSING_SENTINEL = -1

STATIC_DROP_FOR_CAUSALITY = [
    # In-hospital complication context, confirmed limited to first 48h of admission -
    # not safe as a confounder across all cutoff variants (24h/48h/72h)
    'DIABETIC_COMA', 'NEUROVASCULAR_COMPLICATION', 'HEMIPLEGIA', 'SKIN_ULCER',
    'RETINOPATHY', 'ACUTE_RESPIRATORY_DISORDER', 'UNEXPECTED_FALLS',
    'OTHER_COMPLICATION', 'BYPASS', 'ATHEROSCLEROSIS',
    # In-hospital medication administration - too close to the treatment being evaluated
    'SGLT2_HOSPITAL_BITZUA', 'METFORMIN_HOSPITAL_BITZUA',
    'ANTIDIABETIC_HIGH_HYPO_HOSPITAL_BITZUA',
]


def flip_missing_sentinel(df: pd.DataFrame, id_cols: List[str] = STATIC_ID_COLS, sentinel=MISSING_SENTINEL) -> pd.DataFrame:
    """Convert the -1 missing-value placeholder to real NaN across all numeric columns."""
    df = df.copy()
    numeric_cols = [c for c in df.columns if c not in id_cols and pd.api.types.is_numeric_dtype(df[c])]
    for col in numeric_cols:
        df.loc[df[col] == sentinel, col] = np.nan
    return df


def drop_for_causality(df: pd.DataFrame, drop_cols: List[str] = STATIC_DROP_FOR_CAUSALITY) -> pd.DataFrame:
    """Remove columns structurally unsafe for the causal claim (post-treatment bias, leakage)."""
    present = [c for c in drop_cols if c in df.columns]
    if present:
        print(f"[drop_for_causality] Dropping for causal validity: {present}")
    return df.drop(columns=present)


static_df = flip_missing_sentinel(static_df)
static_df = drop_for_causality(static_df)

static_df.to_csv(RESULTS_DIR / 'context_static_input_clean.csv', index=False)
print(f"Rows: {len(static_df)}, Columns: {static_df.shape[1]}")
column_summary(static_df)

In [ ]:
def to_utc_naive(s: pd.Series) -> pd.Series:
    """
    Ensures datetimes are comparable:
    - If tz-aware: convert to UTC then drop tz
    - If tz-naive: leave as-is
    """
    s = pd.to_datetime(s, errors="coerce")
    if hasattr(s.dt, "tz"):
        # tz-aware or tz-naive both have .dt.tz attribute
        try:
            return s.dt.tz_convert("UTC").dt.tz_localize(None)
        except TypeError:
            # tz-naive, tz_convert will fail
            return s
    return s


def normalize_inputs(output_df, qa_df, static_df, visit_master, cutoff_date=pd.Timestamp("2022-07-01")):
    out = output_df.copy()
    qa = qa_df.copy()
    stat = static_df.copy()
    vm = visit_master.copy()

    # unify VisitId naming FIRST
    if "PatientId" in out.columns:
        out = out.rename(columns={"PatientId": "VisitId"})
    if "PatientId" in qa.columns:
        qa = qa.rename(columns={"PatientId": "VisitId"})
    if "visit_id" in stat.columns:
        stat = stat.rename(columns={"visit_id": "VisitId"})
    if "visit_id" in vm.columns:
        vm = vm.rename(columns={"visit_id": "VisitId"})

    # normalize all datetime columns ONCE, centrally
    for c in ["StartDateTime", "EndDateTime"]:
        if c in out.columns:
            out[c] = to_utc_naive(out[c])
        if c in qa.columns:
            qa[c] = to_utc_naive(qa[c])
        if c in vm.columns:
            vm[c] = to_utc_naive(vm[c])

    out = out[out["StartDateTime"] >= cutoff_date]
    qa = qa[qa["StartDateTime"] >= cutoff_date]
    vm = vm[vm["StartDateTime"] >= cutoff_date]
    return out, qa, stat, vm


# ------------------------------------------------
# 1) Complication Label Logic
# ------------------------------------------------
def add_complication_name_column(output_df: pd.DataFrame) -> pd.DataFrame:
    df = output_df.copy()
    if "Value" not in df.columns:
        df["Value"] = np.nan

    df["ComplicationName"] = df["ConceptName"].astype(str)
    mask = df["ConceptName"].eq("DISGLYCEMIA_EVENT")
    df.loc[mask, "ComplicationName"] = (
        df.loc[mask, "ConceptName"].astype(str) + "__" + df.loc[mask, "Value"].astype(str)
    )
    return df


# ------------------------------------------------
# 2) Visit times: admit + end-of-admission
# ------------------------------------------------
def build_visit_times_from_master(
    visit_master: pd.DataFrame,
    admit_col: str = "StartDateTime",
    duration_col: str = "duration_days"
) -> pd.DataFrame:
    """
    Returns one row per VisitId:
    VisitId, admit_time, duration_days, end_time

    This is used to limit the interesting duration of a visit, and also as a signal for the negative sampling.
    """
    vm = visit_master[["VisitId", admit_col, duration_col]].copy()
    vm = vm.rename(columns={admit_col: "admit_time", duration_col: "duration_days"})

    # enforce types (should already be ok after normalize_inputs, but keep defensive)
    vm["admit_time"] = pd.to_datetime(vm["admit_time"], errors="coerce")
    vm["duration_days"] = pd.to_numeric(vm["duration_days"], errors="coerce")

    vm["end_time"] = vm["admit_time"] + pd.to_timedelta(vm["duration_days"], unit="D")

    vm = vm.dropna(subset=["VisitId", "admit_time", "end_time"])
    return vm


# ------------------------------------------------
# 3) Complication labels: excluded / positive / negative by cutoff_hours
# ------------------------------------------------
def build_complication_labels(
    output_df: pd.DataFrame,
    visit_times: pd.DataFrame,
    complication_concepts: set,
    cutoff_hours: float = 48.0
) -> pd.DataFrame:
    """
    Label each (VisitId, ComplicationName) as positive, negative, or excluded.
    - Occurs strictly before cutoff_hours -> excluded (onset not attributable to in-stay care)
    - Occurs at/after cutoff_hours -> y=1, cutoff_time = first qualifying occurrence
    - Never occurs -> y=0, cutoff_time = end_of_admission
    """
    df = output_df.copy()
    df = df[df["ConceptName"].isin(complication_concepts)].copy()
    df = add_complication_name_column(df)
    df = df.merge(visit_times[["VisitId", "admit_time", "end_time"]], on="VisitId", how="left")
    df = df.dropna(subset=["StartDateTime", "admit_time"])
    df["hours_from_admit"] = (df["StartDateTime"] - df["admit_time"]).dt.total_seconds() / 3600.0

    complication_types = sorted(df["ComplicationName"].unique().tolist())
    all_visits = visit_times[["VisitId", "admit_time", "end_time"]].copy()

    rows = []
    for comp in complication_types:
        comp_events = df[df["ComplicationName"] == comp]
        early_visits = set(comp_events.loc[comp_events["hours_from_admit"] < cutoff_hours, "VisitId"])

        late_events = comp_events[comp_events["hours_from_admit"] >= cutoff_hours]
        first_late = (
            late_events.sort_values("StartDateTime")
            .groupby("VisitId", as_index=False)
            .first()
        )
        pos_visits = set(first_late["VisitId"])

        eligible = all_visits[~all_visits["VisitId"].isin(early_visits)].copy()

        pos = eligible[eligible["VisitId"].isin(pos_visits)].merge(
            first_late[["VisitId", "StartDateTime"]].rename(columns={"StartDateTime": "cutoff_time"}),
            on="VisitId", how="left"
        )
        pos["ComplicationName"] = comp
        pos["y"] = 1

        neg = eligible[~eligible["VisitId"].isin(pos_visits)].copy()
        neg["cutoff_time"] = neg["end_time"]
        neg["ComplicationName"] = comp
        neg["y"] = 0

        rows.append(pos[["VisitId", "ComplicationName", "admit_time", "cutoff_time", "y"]])
        rows.append(neg[["VisitId", "ComplicationName", "admit_time", "cutoff_time", "y"]])

    return pd.concat(rows, ignore_index=True)


# ------------------------------------------------
# 4) QA aspects and windowed mean aggregation
# ------------------------------------------------
def infer_qa_aspects(qa_df: pd.DataFrame) -> List[str]:
    """
    Goal:
    find all qa_aspects = unique(PatternName, ComplianceType) pairs
    """
    tmp = qa_df.copy()
    tmp["aspect"] = tmp["PatternName"].astype(str) + "__" + tmp["ComplianceType"].astype(str)
    aspects = sorted(tmp["aspect"].dropna().unique().tolist())
    return aspects


def aggregate_qa_means_for_window(
    qa_visit: pd.DataFrame,
    admit_time: pd.Timestamp,
    cutoff_time: pd.Timestamp,
    qa_aspects: List[str]
) -> dict:
    """
    Mean ComplianceScore per aspect for QA rows in [admit_time, cutoff_time).
    Method: average within (aspect, day) first, then average across days - so a day
            with many instances of the same pattern doesn't outweigh a day with just
            one (same logic as the earlier daily-adherence analysis).
    """
    if qa_visit is None or qa_visit.empty:
        return {a: np.nan for a in qa_aspects}

    q = qa_visit.dropna(subset=["StartDateTime"]).copy()
    q = q[(q["StartDateTime"] >= admit_time) & (q["StartDateTime"] < cutoff_time)]
    if q.empty:
        return {a: np.nan for a in qa_aspects}

    q["aspect"] = q["PatternName"].astype(str) + "__" + q["ComplianceType"].astype(str)
    q["Date"] = q["StartDateTime"].dt.date

    daily = q.groupby(["aspect", "Date"])["ComplianceScore"].mean().reset_index()
    means = daily.groupby("aspect")["ComplianceScore"].mean().to_dict()

    return {a: means.get(a, np.nan) for a in qa_aspects}


# ------------------------------------------------
# 4b) Missing-data handling for QA aspect columns
# ------------------------------------------------
def impute_qa_features(flat: pd.DataFrame, qa_aspects: List[str], missing_threshold: float = 0.95):
    """
    Drop QA aspect columns missing above `missing_threshold`; for the rest, add a
    binary "<aspect>__missing" indicator (a pattern never applying can itself be
    informative) then median-impute.
    """
    flat = flat.copy()
    missing_frac = flat[qa_aspects].isna().mean()
    keep_aspects = missing_frac[missing_frac <= missing_threshold].index.tolist()
    dropped = [a for a in qa_aspects if a not in keep_aspects]
    if dropped:
        flat = flat.drop(columns=dropped)

    for a in keep_aspects:
        flat[f"{a}__missing"] = flat[a].isna().astype(int)
        flat[a] = flat[a].fillna(flat[a].median())

    return flat, keep_aspects, dropped


# ------------------------------------------------
# 5) Build flat table: every eligible visit, no sampling
# ------------------------------------------------
def build_flat_table(
    output_df: pd.DataFrame,
    qa_df: pd.DataFrame,
    static_df: pd.DataFrame,
    visits_master: pd.DataFrame,
    complication_concepts: set,
    cutoff_hours: float = 48.0,
) -> pd.DataFrame:
    """
    Build a flat supervised learning dataset with unit: (VisitId, ComplicationName).
    Every eligible visit is used as either a positive or a negative for each complication
    (no subsampling) - visits where the complication occurred before `cutoff_hours` are
    excluded, since onset can't be confidently attributed to in-stay treatment there.
    """
    visit_times = build_visit_times_from_master(visits_master)

    labels = build_complication_labels(
        output_df=output_df,
        visit_times=visit_times,
        complication_concepts=complication_concepts,
        cutoff_hours=cutoff_hours
    )

    qa_aspects = infer_qa_aspects(qa_df)
    qa_by_visit = {vid: g for vid, g in qa_df.groupby("VisitId")}

    rows = []
    for r in tqdm(labels.itertuples(index=False), total=len(labels), desc=f"cutoff={cutoff_hours}h"):
        qa_visit = qa_by_visit.get(r.VisitId, qa_df.iloc[0:0])
        feats = aggregate_qa_means_for_window(qa_visit, r.admit_time, r.cutoff_time, qa_aspects)
        rows.append({
            "VisitId": r.VisitId,
            "ComplicationName": r.ComplicationName,
            "cutoff_time": r.cutoff_time,
            "y": r.y,
            **feats
        })
    flat = pd.DataFrame(rows)

    flat, kept_aspects, dropped_aspects = impute_qa_features(flat, qa_aspects)
    if dropped_aspects:
        print(f"[cutoff={cutoff_hours}h] Dropped {len(dropped_aspects)} QA aspects for excess missingness: {dropped_aspects}")

    flat = flat.merge(static_df, on="VisitId", how="left")

    const_cols = [c for c in flat.columns if flat[c].nunique() <= 1]
    flat = flat.drop(columns=const_cols)

    return flat

In [ ]:
output_df, qa_df, static_df, visit_master = normalize_inputs(output_df, qa_df, static_df, visit_master)

CUTOFF_OPTIONS_HOURS = [24, 48, 72]
flat_tables = {}

for cutoff in CUTOFF_OPTIONS_HOURS:
    flat_variant = build_flat_table(
        output_df=output_df,
        qa_df=qa_df,
        static_df=static_df,
        visits_master=visit_master,
        complication_concepts=complication_concepts,
        cutoff_hours=cutoff,
    )
    flat_variant.to_csv(RESULTS_DIR / f'base_ML_data_{cutoff}h.csv', index=False)
    flat_tables[cutoff] = flat_variant
    print(f"[{cutoff}h] shape={flat_variant.shape}, "
          f"y_counts={flat_variant['y'].value_counts().to_dict()}, "
          f"n_complications={flat_variant['ComplicationName'].nunique()}")

summary_df = column_summary(flat_tables[48])
summary_df

In [ ]:
# ============================================================
# Setup
# ============================================================
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn.linear_model"
)
import pandas as pd
import numpy as np

import os
os.environ["MPLCONFIGDIR"] = str(RESULTS_DIR / "matplotlib")
import matplotlib.pyplot as plt

TARGET_COL = "y"
ID_COLS = ["VisitId", "ComplicationName", "cutoff_time", "person_id"]

# Reload from disk if the kernel was restarted and flat_tables isn't in memory
CUTOFF_OPTIONS_HOURS = [24, 48, 72]
if "flat_tables" not in globals():
    print("[Setup] flat_tables not found in memory - reloading from cached CSVs")
    flat_tables = {}
    for cutoff in CUTOFF_OPTIONS_HOURS:
        path = RESULTS_DIR / f'base_ML_data_{cutoff}h.csv'
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{path} not found - rerun the data-build cell (Cell 3) to regenerate it."
            )
        flat_tables[cutoff] = pd.read_csv(path)
        print(f"  loaded {path}: shape={flat_tables[cutoff].shape}")

# Union of columns across all cutoff variants, so a feature present in one
# variant but not another still gets picked up by the per-run filtering below
ALL_FLAT_COLUMNS = set()
for _fdf in flat_tables.values():
    ALL_FLAT_COLUMNS.update(_fdf.columns)

QA_FEATURES = sorted([c for c in ALL_FLAT_COLUMNS if "__" in c])
STATIC_FEATURES = sorted([
    c for c in ALL_FLAT_COLUMNS
    if c not in ID_COLS + [TARGET_COL] + QA_FEATURES
])

print("QA features:", len(QA_FEATURES))
print("Static features:", len(STATIC_FEATURES))

def preprocess_missing(X: pd.DataFrame, qa_features, static_features):
    X = X.copy()
    X[static_features] = X[static_features].replace(-1, np.nan)
    return X

qa_imputer = SimpleImputer(strategy="mean")
static_imputer = SimpleImputer(strategy="median")

In [ ]:
# ============================================================
# Cell 1 (patched): Per-complication Firth (bias-reduced) logistic
# regression with real p-values, FDR-corrected significance, a
# lightweight stability check, and circularity-aware QA exclusion
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import statsmodels.api as sm
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

# Patterns whose compliance gating is definitionally entangled with specific
# complication concepts - using them to predict those same complications risks
# circularity, not just confounding. Confirmed via the pattern XML (context gate
# references the same event concept as the outcome), not excluded blanket-wide.
QA_EXCLUDE_FOR_COMPLICATION = {
    "HYPERGLYCEMIA_EVENT": ["ROUTINE_GLUCOSE_MEASURE_PATTERN"],
    "SEVERE_HYPERGLYCEMIA_EVENT": ["ROUTINE_GLUCOSE_MEASURE_PATTERN"],
    "HYPOGLYCEMIA_EVENT": ["ROUTINE_GLUCOSE_MEASURE_PATTERN"],
    "SEVERE_HYPOGLYCEMIA_EVENT": ["ROUTINE_GLUCOSE_MEASURE_PATTERN"],
}


def fit_firth_logistic(X: np.ndarray, y: np.ndarray, max_iter: int = 500, tol: float = 1e-6) -> dict:
    """
    Purpose: Firth's bias-reduced logistic regression, implemented directly since
             the `firthlogist` package requires Python <3.11 (this env runs 3.13).
    Method: Standard Firth (1993) algorithm - IRLS with a Jeffreys-prior penalty
            added to the score equations, which keeps the likelihood surface
            well-behaved under separation/near-collinearity (unlike plain MLE,
            which diverges) while still producing a proper information matrix
            for Wald-style p-values.

    Args:
        X (np.ndarray): Design matrix, shape (n, p), first column must be the intercept.
        y (np.ndarray): Binary outcome, shape (n,).
        max_iter (int): Maximum IRLS iterations.
        tol (float): Convergence tolerance on the coefficient update norm.

    Returns:
        dict: beta, se, pvals, ci_lower, ci_upper, converged, n_iter
    """
    n, p = X.shape
    beta = np.zeros(p)
    converged = False
    n_iter_used = max_iter

    for i in range(max_iter):
        eta = X @ beta
        pi = np.clip(1 / (1 + np.exp(-eta)), 1e-10, 1 - 1e-10)
        W = pi * (1 - pi)

        XtWX = (X * W[:, None]).T @ X
        try:
            XtWX_inv = np.linalg.inv(XtWX)
        except np.linalg.LinAlgError:
            XtWX_inv = np.linalg.pinv(XtWX)

        # Hat matrix diagonal: h_i = W_i * x_i' (X'WX)^-1 x_i
        WX = X * np.sqrt(W)[:, None]
        H_diag = np.einsum('ij,jk,ik->i', WX, XtWX_inv, WX)

        # Firth-modified score: adds h_i*(0.5 - pi_i) bias correction
        U_star = X.T @ (y - pi + H_diag * (0.5 - pi))
        delta = XtWX_inv @ U_star
        beta = beta + delta

        if np.max(np.abs(delta)) < tol:
            converged = True
            n_iter_used = i + 1
            break

    # Final covariance from the penalized fit's information matrix
    eta = X @ beta
    pi = np.clip(1 / (1 + np.exp(-eta)), 1e-10, 1 - 1e-10)
    W = pi * (1 - pi)
    XtWX = (X * W[:, None]).T @ X
    try:
        cov = np.linalg.inv(XtWX)
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(XtWX)

    se = np.sqrt(np.diag(cov))
    z = beta / se
    pvals = 2 * (1 - norm.cdf(np.abs(z)))

    return {
        "beta": beta,
        "se": se,
        "pvals": pvals,
        "ci_lower": beta - 1.96 * se,
        "ci_upper": beta + 1.96 * se,
        "converged": converged,
        "n_iter": n_iter_used,
    }


def _fit_firth(X_design, y, maxiter=500):
    """Fit Firth logistic regression, suppressing library-level numeric warnings."""
    with warnings.catch_warnings(), np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        warnings.simplefilter("ignore")
        return fit_firth_logistic(X_design.values, y.values, max_iter=maxiter)


def bootstrap_sign_stability(X_design, y, point_signs, n_boot=50, random_state=42, maxiter=50):
    """
    Purpose: Lightweight stability check - what fraction of resampled fits agree
             in sign with the full-data point estimate. Supplementary to the Wald
             p-value, not a replacement - flags fragile estimates the single fit
             might not reveal (small-sample bias, near-separation). Uses Firth for
             the resamples too, for the same separation-robustness reason.
    """
    rng = np.random.default_rng(random_state)
    n = len(y)
    agree_counts = np.zeros(len(point_signs))
    successful = 0

    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        Xb, yb = X_design.iloc[idx], y.iloc[idx]
        if yb.nunique() < 2:
            continue
        try:
            fit_b = _fit_firth(Xb, yb, maxiter=maxiter)
            boot_signs = np.sign(fit_b["beta"][1:])  # drop intercept
            agree_counts += (boot_signs == point_signs)
            successful += 1
        except Exception:
            continue

    if successful == 0:
        return np.full(len(point_signs), np.nan)
    return agree_counts / successful


def run_logit_feature_importance_by_complication(
    flat_df: pd.DataFrame,
    complications: list,
    QA_FEATURES: list,
    STATIC_FEATURES: list,
    TARGET_COL: str,
    min_positives: int = 30,
    corr_threshold: float = 0.8,
    maxiter: int = 500,
    n_boot_stability: int = 50,
    fdr_alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Purpose: Per-complication logistic regression giving real p-values, FDR-corrected
             significance, and confidence intervals, using Firth's bias-reduced MLE -
             plain unregularized MLE failed on 8/10 complications with Singular
             matrix / non-convergence errors from separation, once negative sampling
             was removed and every complication's model uses the full eligible-visit
             population. Firth is the standard fix for exactly this (rare/moderate-
             event logistic regression with many covariates) while still yielding
             valid, interpretable p-values.
    Method: QA and STATIC features sit together in ONE model so the adherence
            coefficient is interpreted "holding measured severity/comorbidity
            constant" - this is the actual confounding-by-indication adjustment.
            QA aspects definitionally entangled with the outcome's own concept
            (per QA_EXCLUDE_FOR_COMPLICATION) are dropped before fitting. Benjamini-
            Hochberg FDR correction is applied within each complication's own set
            of tested features (the natural "family" of simultaneous tests in one
            fit) - correlation-pruned features are excluded from the correction
            family since they were never actually tested. A lightweight bootstrap
            sign-agreement check supplements the p-value.
    """
    all_results = []

    def _prune_corr(X_sub: pd.DataFrame, threshold: float):
        if X_sub.shape[1] <= 1:
            return X_sub, []
        corr = X_sub.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop = [c for c in upper.columns if (upper[c] > threshold).any()]
        return X_sub.drop(columns=drop), drop

    for comp in tqdm(complications, desc="Complications"):
        df = flat_df[flat_df["ComplicationName"] == comp].copy()
        if df.empty or TARGET_COL not in df.columns:
            print(f"[{comp}] skipped (empty or missing target)")
            continue

        y = df[TARGET_COL].astype(int)
        n_pos = int(y.sum())
        if n_pos < min_positives:
            print(f"[{comp}] skipped (n_pos={n_pos} < {min_positives})")
            continue

        excluded_patterns = QA_EXCLUDE_FOR_COMPLICATION.get(comp, [])
        qa_feats = [
            f for f in QA_FEATURES
            if f in df.columns and f.split("__")[0] not in excluded_patterns
        ]
        if excluded_patterns:
            print(f"[{comp}] excluding QA aspects from {excluded_patterns} (circular with outcome definition)")

        stat_feats = [f for f in STATIC_FEATURES if f in df.columns]
        feats = qa_feats + stat_feats
        if not feats:
            print(f"[{comp}] skipped (no usable features)")
            continue

        X = df[feats].apply(pd.to_numeric, errors="coerce")
        X_qa = X[qa_feats].copy() if qa_feats else pd.DataFrame(index=X.index)
        X_st = X[stat_feats].copy() if stat_feats else pd.DataFrame(index=X.index)

        qa_all_missing = X_qa.columns[X_qa.isna().all()].tolist() if not X_qa.empty else []
        st_all_missing = X_st.columns[X_st.isna().all()].tolist() if not X_st.empty else []
        X_qa = X_qa.drop(columns=qa_all_missing) if qa_all_missing else X_qa
        X_st = X_st.drop(columns=st_all_missing) if st_all_missing else X_st

        if not X_qa.empty:
            X_qa = pd.DataFrame(qa_imputer.fit_transform(X_qa), columns=X_qa.columns, index=X_qa.index)
        if not X_st.empty:
            X_st = pd.DataFrame(static_imputer.fit_transform(X_st), columns=X_st.columns, index=X_st.index)

        X = pd.concat([X_qa, X_st], axis=1)
        qa_feats = [c for c in qa_feats if c in X.columns]
        stat_feats = [c for c in stat_feats if c in X.columns]
        X = X[qa_feats + stat_feats]

        var = X.var(axis=0)
        X = X[var[var > 1e-5].index.tolist()]

        qa_keep = [c for c in X.columns if c in qa_feats]
        st_keep = [c for c in X.columns if c in stat_feats]
        X_qa, dropped_qa = _prune_corr(X[qa_keep], corr_threshold) if qa_keep else (X.iloc[:, 0:0], [])
        X_st, dropped_st = _prune_corr(X[st_keep], corr_threshold) if st_keep else (X.iloc[:, 0:0], [])
        X = pd.concat([X_qa, X_st], axis=1)

        final_features = X.columns.tolist()
        if not final_features:
            print(f"[{comp}] skipped (all features removed by filters)")
            continue

        # Standardize so coefficients are comparable across features
        X_scaled = (X - X.mean()) / X.std(ddof=0).replace(0, 1)
        X_design = sm.add_constant(X_scaled, has_constant="add")

        try:
            fit = _fit_firth(X_design, y, maxiter=maxiter)
        except Exception as e:
            print(f"[{comp}] SKIPPED - fit failed: {e}")
            continue

        if not fit["converged"]:
            print(f"[{comp}] SKIPPED - did not converge within maxiter={maxiter}")
            continue

        # Global sanity check only - NOT evidence for any single coefficient's validity,
        # just "did this model learn anything at all"
        pi_full = 1 / (1 + np.exp(-(X_design.values @ fit["beta"])))
        auc = roc_auc_score(y, pi_full) if y.nunique() == 2 else np.nan

        coefs = fit["beta"][1:]       # drop intercept
        pvals = fit["pvals"][1:]
        ci_lower = fit["ci_lower"][1:]
        ci_upper = fit["ci_upper"][1:]

        # FDR correction within this complication's own family of tested features -
        # only the features actually fit, NOT the correlation-pruned ones (those
        # were never tested and shouldn't dilute or pad the correction family)
        _, pvals_fdr, _, _ = multipletests(pvals, alpha=fdr_alpha, method="fdr_bh")
        significant_fdr = pvals_fdr < fdr_alpha

        point_signs = np.sign(coefs)
        sign_agreement = bootstrap_sign_stability(X_design, y, point_signs, n_boot=n_boot_stability)

        pruned = dropped_qa + dropped_st
        comp_df = pd.DataFrame({
            "feature": final_features,
            "coef": coefs,
            "p_value": pvals,
            "p_value_fdr": pvals_fdr,
            "significant_fdr": significant_fdr,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "sign_agreement": sign_agreement,
            "feature_type": ["QA" if f in qa_feats else "STATIC" for f in final_features],
            "dropped_by_corr": False,
            "complication": comp,
            "n_pos": n_pos,
            "n_samples": int(len(X)),
            "n_features_after_prune": len(final_features),
            "converged": True,
            "auc": auc,
        })

        if pruned:
            pruned_df = pd.DataFrame({
                "feature": pruned,
                "coef": 0.0,
                "p_value": 1.0,
                "p_value_fdr": 1.0,
                "significant_fdr": False,
                "ci_lower": 0.0,
                "ci_upper": 0.0,
                "sign_agreement": np.nan,
                "feature_type": ["QA" if f in qa_feats else "STATIC" for f in pruned],
                "dropped_by_corr": True,
                "complication": comp,
                "n_pos": n_pos,
                "n_samples": int(len(X)),
                "n_features_after_prune": len(final_features),
                "converged": True,
                "auc": auc,
            })
            comp_df = pd.concat([comp_df, pruned_df], ignore_index=True)

        comp_df = comp_df.sort_values("p_value_fdr")
        all_results.append(comp_df)

        n_sig = int(significant_fdr.sum())
        print(f"[{comp}] done | n_pos={n_pos} | n={len(X)} | features={len(final_features)} | "
              f"pruned={len(pruned)} | auc={auc:.3f} | significant_fdr={n_sig}/{len(final_features)}")

    if not all_results:
        return pd.DataFrame()
    return pd.concat(all_results, ignore_index=True)


In [ ]:
# ============================================================
# Cell 2 (patched): Multi-cutoff wrapper - robust now requires
# FDR-significant AND sign-consistent in ALL cutoff variants
# ============================================================
def run_logit_importance_multi_cutoff(
    flat_tables: dict,
    QA_FEATURES: list,
    STATIC_FEATURES: list,
    TARGET_COL: str,
    **kwargs
) -> pd.DataFrame:
    """
    Purpose: Run the logistic-regression feature importance separately on each
             cutoff-hour dataset, then flag features FDR-significant AND
             sign-consistent in ALL cutoff variants where the complication
             actually converged - the combined bar for a defensible finding:
             controlled false-discovery rate at the individual-fit level, plus
             replication across the three temporal cutoff choices.
    """
    all_runs = []
    for cutoff_hours, flat_df in flat_tables.items():
        print(f"\n=== cutoff_hours={cutoff_hours} ===")
        result = run_logit_feature_importance_by_complication(
            flat_df=flat_df,
            complications=sorted(flat_df["ComplicationName"].unique()),
            QA_FEATURES=QA_FEATURES,
            STATIC_FEATURES=STATIC_FEATURES,
            TARGET_COL=TARGET_COL,
            **kwargs
        )
        result["cutoff_hours"] = cutoff_hours
        all_runs.append(result)

    combined = pd.concat(all_runs, ignore_index=True)

    n_cutoffs = len(flat_tables)

    def _robust(group):
        if len(group) < n_cutoffs:
            return False  # didn't converge / wasn't tested in every cutoff variant
        if not group["significant_fdr"].all():
            return False
        signs = np.sign(group["coef"])
        return bool((signs == signs.iloc[0]).all()) and signs.iloc[0] != 0

    robust_flags = (
        combined.groupby(["complication", "feature"])
        .apply(_robust)
        .rename("robust_across_cutoffs")
        .reset_index()
    )
    combined = combined.merge(robust_flags, on=["complication", "feature"], how="left")

    # Clear, single "is this a defensible finding" marker: significant within its own
    # model's FDR-corrected family AND replicated across all cutoff variants
    combined["final_significant"] = combined["significant_fdr"] & combined["robust_across_cutoffs"]

    return combined


importance_df_all = run_logit_importance_multi_cutoff(
    flat_tables=flat_tables,
    QA_FEATURES=QA_FEATURES,
    STATIC_FEATURES=STATIC_FEATURES,
    TARGET_COL=TARGET_COL,
)

importance_df_all.to_csv(RESULTS_DIR / "importance_df_all.csv", index=False)

print(f"Rows: {len(importance_df_all)}, "
      f"Complications: {importance_df_all['complication'].nunique()}, "
      f"Cutoffs: {sorted(importance_df_all['cutoff_hours'].unique())}, "
      f"Final significant rows: {int(importance_df_all['final_significant'].sum())}")


In [ ]:
# ============================================================
# Cutoff-averaged summary - add min_p_value_fdr (best case across cutoffs)
# ============================================================
importance_avg_all = (
    importance_df_all
    .groupby(["complication", "feature", "feature_type"])
    .agg(
        mean_coef=("coef", "mean"),
        abs_mean_coef=("coef", lambda s: s.abs().mean()),
        mean_p_value_fdr=("p_value_fdr", "mean"),
        min_p_value_fdr=("p_value_fdr", "min"),   # significant (FDR) in at least 1 cutoff
        max_p_value_fdr=("p_value_fdr", "max"),   # significant (FDR) in ALL cutoffs
        mean_sign_agreement=("sign_agreement", "mean"),
    )
    .reset_index()
)

robust_pairs = set(
    zip(
        importance_df_all.loc[importance_df_all["robust_across_cutoffs"], "complication"],
        importance_df_all.loc[importance_df_all["robust_across_cutoffs"], "feature"],
    )
)
importance_avg_all["robust_across_cutoffs"] = importance_avg_all.apply(
    lambda r: (r["complication"], r["feature"]) in robust_pairs, axis=1
)


In [ ]:
# ============================================================
# Top-K features per complication - FDR-significant in at least one
# cutoff variant; "*" marks the stricter robust-in-all-cutoffs set
# ============================================================
FDR_ALPHA = 0.05
top_k = 15

importance_significant_any = importance_avg_all[importance_avg_all["min_p_value_fdr"] < FDR_ALPHA].copy()

top_features_per_comp = (
    importance_significant_any
    .sort_values(
        ["complication", "min_p_value_fdr", "abs_mean_coef"],
        ascending=[True, True, False]
    )
    .groupby("complication")
    .head(top_k)
)

top_features_per_comp["feature_label"] = top_features_per_comp.apply(
    lambda r: f"{r['feature']}*" if r["robust_across_cutoffs"] else r["feature"], axis=1
)

top_features_per_comp[
    ["complication", "feature_label", "feature_type", "mean_coef", "abs_mean_coef",
     "min_p_value_fdr", "max_p_value_fdr", "robust_across_cutoffs", "mean_sign_agreement"]
].to_csv(RESULTS_DIR / 'top_features_per_comp.csv', index=False)

top_features_per_comp[["complication", "feature_label", "feature_type", "mean_coef", "min_p_value_fdr", "robust_across_cutoffs"]]


In [ ]:
qa_only = top_features_per_comp[top_features_per_comp["feature_type"] == "QA"]
qa_only[["complication", "feature_label", "mean_coef", "min_p_value_fdr", "robust_across_cutoffs", "mean_sign_agreement"]]


In [ ]:
# ============================================================
# Top-K static (confounder) features per complication - FDR-significant
# in at least one cutoff variant; "*" marks robust-in-all-cutoffs
# ============================================================
top_k_static = 5

top_static_per_comp = (
    importance_significant_any
    .query("feature_type == 'STATIC'")
    .sort_values(["complication", "min_p_value_fdr", "abs_mean_coef"], ascending=[True, True, False])
    .groupby("complication")
    .head(top_k_static)
)

top_static_per_comp["feature_label"] = top_static_per_comp.apply(
    lambda r: f"{r['feature']}*" if r["robust_across_cutoffs"] else r["feature"], axis=1
)

top_static_per_comp[
    ["complication", "feature_label", "mean_coef", "min_p_value_fdr", "robust_across_cutoffs", "mean_sign_agreement"]
]


In [ ]:
# ============================================================
# QA vs STATIC contribution split (based on full averaged magnitude,
# not just the significant subset - measures overall explanatory weight,
# not just what cleared the significance bar)
# ============================================================
importance_summary = (
    importance_avg_all
    .groupby(["complication", "feature_type"])["abs_mean_coef"]
    .sum()
    .reset_index()
    .pivot(index="complication", columns="feature_type", values="abs_mean_coef")
    .fillna(0)
)

importance_summary["QA_fraction"] = (
    importance_summary.get("QA", 0)
    / (importance_summary.get("QA", 0) + importance_summary.get("STATIC", 0) + 1e-12)
)

importance_summary.sort_values("QA_fraction", ascending=False)


In [ ]:
# ============================================================
# Plot: top QA and STATIC features side by side - FDR-significant in
# at least one cutoff variant, robust-in-all-cutoffs ones starred
# ============================================================
def plot_top_features(comp, k=8, df=None, robust_features=None):
    """
    Purpose: Plot top-k QA and STATIC features side by side for a given complication.
    Method: Two horizontal bar subplots, each restricted to its feature_type and
            filtered to features FDR-significant in at least one cutoff variant,
            colored by a fixed categorical slot. Features robust across ALL cutoff
            variants get a "*" marker on their label.

    Args:
        comp (str): Complication name to filter on.
        k (int): Number of top features to show per panel.
        df (pd.DataFrame): Significant-features summary (min_p_value_fdr already filtered).
        robust_features (set): (complication, feature) pairs considered robust.

    Returns:
        None: Displays the figure.
    """
    if df is None:
        df = importance_significant_any
    if robust_features is None:
        robust_features = set(zip(
            importance_avg_all.loc[importance_avg_all["robust_across_cutoffs"], "complication"],
            importance_avg_all.loc[importance_avg_all["robust_across_cutoffs"], "feature"],
        ))

    d_comp = df[df["complication"] == comp].copy()

    feature_types = ["QA", "STATIC"]
    colors = {"QA": "#2a78d6", "STATIC": "#1baf7a"}  # fixed categorical slots, never swapped

    fig, axes = plt.subplots(1, len(feature_types), figsize=(14, 4), sharex=False)

    for ax, ftype in zip(axes, feature_types):
        d = d_comp[d_comp["feature_type"] == ftype].sort_values(
            ["min_p_value_fdr", "abs_mean_coef"],
            ascending=[True, False]
        ).head(k)

        labels = [f"{f}*" if (comp, f) in robust_features else f for f in d["feature"]]

        ax.barh(labels, d["mean_coef"], color=colors[ftype])
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(ftype)

    fig.suptitle(f"Top significant features for {comp}  (* = robust across all cutoff variants)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Plot: top QA and STATIC features side by side, robust ones starred
# ============================================================
def plot_top_features(comp, k=8, df=None, robust_features=None):
    """
    Purpose: Plot top-k QA and STATIC features side by side for a given complication.
    Method: Two horizontal bar subplots, each restricted to its feature_type, colored
            by a fixed categorical slot. Features robust across all cutoff variants
            get a "*" marker on their label.

    Args:
        comp (str): Complication name to filter on.
        k (int): Number of top features to show per panel.
        df (pd.DataFrame): Cutoff-averaged feature importance dataframe.
        robust_features (set): (complication, feature) pairs considered robust.

    Returns:
        None: Displays the figure.
    """
    if df is None:
        df = importance_avg_all
    if robust_features is None:
        robust_features = set(zip(importance_robust_summary["complication"], importance_robust_summary["feature"]))

    d_comp = df[df["complication"] == comp].copy()

    feature_types = ["QA", "STATIC"]
    colors = {"QA": "#2a78d6", "STATIC": "#1baf7a"}  # fixed categorical slots, never swapped

    fig, axes = plt.subplots(1, len(feature_types), figsize=(14, 4), sharex=False)

    for ax, ftype in zip(axes, feature_types):
        d = d_comp[d_comp["feature_type"] == ftype].sort_values(
            ["max_p_value", "abs_mean_coef"],
            ascending=[True, False]
        ).head(k)

        labels = [f"{f}*" if (comp, f) in robust_features else f for f in d["feature"]]

        ax.barh(labels, d["mean_coef"], color=colors[ftype])
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(ftype)

    fig.suptitle(f"Top features for {comp}  (* = robust across all cutoff variants)")
    plt.tight_layout()
    plt.show()


plot_top_features("DEATH_EVENT", k=15)


In [ ]:
# ============================================================
# Positive-case counts per complication, per cutoff variant,
# plus feature-selection summary from the importance pipeline
# ============================================================
comp_counts = pd.concat([
    fdf.loc[fdf[TARGET_COL] == 1, "ComplicationName"]
    .value_counts()
    .rename("n_positive")
    .reset_index()
    .rename(columns={"index": "ComplicationName"})
    .assign(cutoff_hours=cutoff)
    for cutoff, fdf in flat_tables.items()
])

comp_counts_pivot = comp_counts.pivot(index="ComplicationName", columns="cutoff_hours", values="n_positive")

# N Selected Features: significant_fdr in at least one cutoff variant
selected_features = (
    importance_df_all[importance_df_all['significant_fdr']]
    .groupby('complication')['feature']
    .nunique()
    .rename('N Selected Features')
)

# N Robust Features: significant + sign-consistent in ALL cutoff variants
robust_features = (
    importance_df_all[importance_df_all['robust_across_cutoffs']]
    .groupby('complication')['feature']
    .nunique()
    .rename('N Robust Features')
)

comp_counts_pivot = comp_counts_pivot.join(selected_features).join(robust_features)
comp_counts_pivot[['N Selected Features', 'N Robust Features']] = (
    comp_counts_pivot[['N Selected Features', 'N Robust Features']].fillna(0).astype(int)
)

comp_counts_pivot


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import to_rgb

FDR_ALPHA = 0.05

COMPLICATION_LABELS = {
    "DEATH_EVENT": "Death",
    "KIDNEY_COMPLICATION_EVENT": "Inpatient AKI",
    "HYPERGLYCEMIA_EVENT": "Hyperglycemia",
    "SEVERE_HYPERGLYCEMIA_EVENT": "Severe Hyperglycemia",
    "HYPOGLYCEMIA_EVENT": "Hypoglycemia",
    "SEVERE_HYPOGLYCEMIA_EVENT": "Severe Hypoglycemia",
    "INFECTION_EVENT": "Infection",
    "HYPEROSMOLALITY_EVENT": "Hyperosmolarity",
    "KETOACIDOSIS_EVENT": "Ketoacidosis",
    "CARDIO-VASCULAR_DISORDER_EVENT": "Inpatient MI",
}

PATTERN_LABELS = {
    "ROUTINE_GLUCOSE_MEASURE_PATTERN": "Routine glucose measurement",
    "ROUTINE_GLUCOSE_MEASURE_ON_ADMISSION_PATTERN": "Routine glucose measurement on admission",
    "ROUTINE_GLUCOSE_MEASURE_ON_STEROIDS_PATTERN": "Routine glucose measurement on steroids",
    "INSULIN_ON_HYPERGLYCEMIA_PATTERN": "Insulin on hyperglycemia",
    "ROUTINE_BASAL_DOSAGE_PATTERN": "Routine basal dosage",
    "CREATININE_MEASURE_ON_ADMISSION": "Creatinine measurement on admission",
    "BMI_MEASURE_ON_ADMISSION": "BMI measurement on admission",
    "REDUCE_BASAL_ON_HYPOGLYCEMIA_PATTERN": "Reduce basal on hypoglycemia",
    "REDUCE_BOLUS_ON_HYPOGLYCEMIA_PATTERN": "Reduce bolus on hypoglycemia",
    "CONTINUE_BASAL_ON_ADMISSION_PATTERN": "Continue basal on admission",
    "CONTINUE_METFORMIN_ON_ADMISSION_PATTERN": "Continue metformin on admission",
    "STOP_ANTIDIABETICS_ON_ADMISSION_PATTERN": "Stop antidiabetics on admission",
    "STOP_ANTIDIABETICS_ON_HYPOGLYCEMIA_PATTERN": "Stop antidiabetics on hypoglycemia",
    "STOP_SGLT2_ON_ADMISSION_PATTERN": "Stop SGLT2 on admission",
}

# 1. QA-only, cutoff-averaged table, one row per (complication, feature)
qa_avg = importance_avg_all[importance_avg_all['feature_type'] == 'QA'].copy()
qa_avg['PatternName'] = qa_avg['feature'].str.split('__').str[0]

# 2. Collapse ComplianceType dimensions -> one cell per (complication, PatternName):
#    the most significant compliance-type variant represents the pattern
qa_avg = qa_avg.sort_values('min_p_value_fdr')
cell_data = (
    qa_avg
    .groupby(['complication', 'PatternName'], as_index=False)
    .first()  # most significant row per (complication, pattern), already sorted above
)

# 3. Significance (>=1 cutoff) + direction + robust (all 3 cutoffs) flags
cell_data['is_significant'] = cell_data['min_p_value_fdr'] < FDR_ALPHA
cell_data['plot_value'] = np.where(cell_data['is_significant'], cell_data['mean_coef'], np.nan)

# 4. Build matrix: rows = pattern, columns = complication
value_matrix = cell_data.pivot(index='PatternName', columns='complication', values='plot_value')
robust_matrix = (
    cell_data.pivot(index='PatternName', columns='complication', values='robust_across_cutoffs')
    .fillna(False)
    .infer_objects(copy=False)
)

# Keep only patterns significant for at least one complication
sig_counts = value_matrix.notna().sum(axis=1)
value_matrix = value_matrix.loc[sig_counts[sig_counts > 0].index]
robust_matrix = robust_matrix.loc[value_matrix.index]

# Sort rows by frequency of significant occurrence, descending - ties broken by
# number of robust (all-cutoff) cells, so the ordering is deliberate, not arbitrary
sig_counts = value_matrix.notna().sum(axis=1)
robust_counts = robust_matrix.sum(axis=1)
row_order = (
    pd.DataFrame({'sig_count': sig_counts, 'robust_count': robust_counts})
    .sort_values(['sig_count', 'robust_count'], ascending=[False, False])
    .index
)
value_matrix = value_matrix.loc[row_order]
robust_matrix = robust_matrix.loc[row_order]

row_labels = [PATTERN_LABELS.get(p, p) for p in value_matrix.index]
col_labels = [COMPLICATION_LABELS.get(c, c) for c in value_matrix.columns]

# 5. Plot - flat categorical colors so every significant cell is equally visible,
#    regardless of coefficient magnitude (recurrence, not effect size, is the message)
SIG_BLUE = to_rgb('#3182bd')
SIG_RED = to_rgb('#de425b')
BLANK_GRAY = to_rgb('#e8e8e8')  # light gray instead of pure white - colored cells pop more

color_matrix = np.tile(np.array(BLANK_GRAY), value_matrix.shape + (1,))
vals = value_matrix.values
color_matrix[vals < 0] = SIG_BLUE
color_matrix[vals > 0] = SIG_RED

fig, ax = plt.subplots(figsize=(max(8, len(col_labels) * 1.1), max(4, len(row_labels) * 0.5)))
ax.imshow(color_matrix, aspect='auto')

# Bold border + star for robust (significant in ALL 3 cutoffs, same direction) cells
for i in range(value_matrix.shape[0]):
    for j in range(value_matrix.shape[1]):
        if robust_matrix.values[i, j] and not pd.isna(value_matrix.values[i, j]):
            rect = patches.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor='black', linewidth=2.5)
            ax.add_patch(rect)
            ax.text(j, i, '\u2605', ha='center', va='center', fontsize=10, color='black', fontweight='bold')

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=45, ha='right')
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels)

# Grid lines between cells for readability
ax.set_xticks(np.arange(-0.5, len(col_labels), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(row_labels), 1), minor=True)
ax.grid(which='minor', color='#ffffff', linewidth=1.2)
ax.tick_params(which='minor', bottom=False, left=False)

ax.set_title("Recurring Guideline Components", fontsize=13, pad=15)

# Legend - plain coefficient-sign language, not "protective/harmful" (that framing
# belongs to the IPTW section, not this observational/predictive-model figure)
legend_handles = [
    patches.Patch(facecolor=SIG_RED, label='Positive coefficient'),
    patches.Patch(facecolor=SIG_BLUE, label='Negative coefficient'),
    patches.Patch(facecolor=BLANK_GRAY, edgecolor='none', label='Not significant'),
    patches.Patch(facecolor='white', edgecolor='black', label='Robust across all cutoffs (\u2605)'),
]
ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8, frameon=False)

fig.text(
    0.01, -0.05,
    "Figure X. Recurring guideline-adherence components identified by sparse regression models across clinical\n"
    "outcomes. Rows correspond to guideline recommendations and columns to clinical outcomes. Colored cells\n"
    "indicate statistically significant associations after FDR correction (red: positive regression coefficient; blue:\n"
    "negative regression coefficient). Stars denote associations that remained significant with a consistent direction\n"
    "across all three observation-window definitions (24, 48, and 72 hours). Blank cells indicate no statistically\n"
    "significant association.",
    fontsize=8, ha='left', va='top'
)

plt.tight_layout()
plt.show()


## Moving from Adherance as Regression Term to Balanced Sickness Covariates

In [ ]:
def build_total_adherence_score(df: pd.DataFrame, qa_feats: list) -> pd.Series:
    """
    Purpose: Aggregate multiple QA aspect columns into one total adherence score
             per visit, weighting each PATTERN equally regardless of how many
             ComplianceType aspects it has (same two-stage logic used earlier in
             the daily-adherence analysis, so one pattern's aspects don't dominate).
    """
    pattern_groups = {}
    for f in qa_feats:
        pattern = f.split("__")[0]
        pattern_groups.setdefault(pattern, []).append(f)

    pattern_scores = pd.DataFrame({
        pattern: df[cols].mean(axis=1)
        for pattern, cols in pattern_groups.items()
    })
    return pattern_scores.mean(axis=1)


def compute_iptw_weights(treatment: pd.Series, X_static: pd.DataFrame, trim_pct: float = 0.01):
    """
    Purpose: Estimate a propensity score P(high adherence | sickness covariates)
             and compute stabilized inverse-probability-of-treatment weights so the
             weighted sample balances measured severity between high/low adherence
             groups - the closest available approximation to "comparing similarly
             sick patients who differ mainly in adherence" without randomization.
    Method: Firth logistic regression for the propensity model (same numerical
            stability reasons as the outcome models), stabilized weights, extreme
            weights trimmed for stability.
    """
    X_design = sm.add_constant(
        (X_static - X_static.mean()) / X_static.std(ddof=0).replace(0, 1),
        has_constant="add"
    )
    fit = _fit_firth(X_design, treatment)
    ps = np.clip(1 / (1 + np.exp(-(X_design.values @ fit["beta"]))), 0.01, 0.99)

    p_treat = treatment.mean()
    weights = np.where(treatment == 1, p_treat / ps, (1 - p_treat) / (1 - ps))

    lo, hi = np.quantile(weights, [trim_pct, 1 - trim_pct])
    return np.clip(weights, lo, hi), ps


def check_covariate_balance(X_static: pd.DataFrame, treatment: pd.Series, weights: np.ndarray) -> pd.DataFrame:
    """
    Purpose: Standardized mean difference (SMD) between high/low adherence groups,
             before and after weighting - the standard diagnostic for whether IPTW
             actually achieved balance. |SMD| < 0.1 is the conventional
             "well balanced" threshold.
    """
    balance = []
    for col in X_static.columns:
        x = X_static[col].values
        pooled_sd = x.std(ddof=0) or 1.0

        smd_before = (x[treatment == 1].mean() - x[treatment == 0].mean()) / pooled_sd

        wm1 = np.average(x[treatment == 1], weights=weights[treatment == 1])
        wm0 = np.average(x[treatment == 0], weights=weights[treatment == 0])
        smd_after = (wm1 - wm0) / pooled_sd

        balance.append({"feature": col, "smd_before": smd_before, "smd_after": smd_after})
    return pd.DataFrame(balance)


def weighted_outcome_test(y: pd.Series, treatment: pd.Series, weights: np.ndarray) -> dict:
    """
    Purpose: After IPTW weighting has balanced measured confounders between
             high/low adherence groups, test whether the outcome still differs -
             the actual "does adherence still differentiate complication risk
             among similarly sick patients" test.
    Method: Weighted logistic regression (y ~ treatment) with robust (sandwich)
            standard errors, since the weights are estimated, not fixed.
    """
    X_design = sm.add_constant(treatment.astype(float), has_constant="add")
    model = sm.GLM(y, X_design, family=sm.families.Binomial(), var_weights=weights)
    with warnings.catch_warnings(), np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        warnings.simplefilter("ignore")
        result = model.fit(cov_type="HC0")

    return {
        "coef": result.params.iloc[1],
        "p_value": result.pvalues.iloc[1],
        "ci_lower": result.conf_int().iloc[1, 0],
        "ci_upper": result.conf_int().iloc[1, 1],
    }


In [ ]:
from statsmodels.stats.multitest import multipletests


def run_iptw_grid_total_score(
    flat_df: pd.DataFrame,
    complications: list,
    QA_FEATURES: list,
    STATIC_FEATURES: list,
    TARGET_COL: str,
    min_positives: int = 30,
) -> pd.DataFrame:
    """
    Purpose: For each complication, test whether TOTAL adherence (high vs low,
             split within that complication's population) still differentiates
             complication probability after IPTW-balancing measured sickness
             covariates. Benjamini-Hochberg FDR correction applied across all
             complications tested in this grid.
    """
    results = []
    for comp in tqdm(complications, desc="IPTW: total score"):
        df = flat_df[flat_df["ComplicationName"] == comp].copy()
        if df.empty:
            continue
        y = df[TARGET_COL].astype(int)
        if y.sum() < min_positives:
            print(f"[{comp}] skipped (n_pos={int(y.sum())} < {min_positives})")
            continue

        excluded_patterns = QA_EXCLUDE_FOR_COMPLICATION.get(comp, [])
        qa_feats = [f for f in QA_FEATURES if f in df.columns and f.split("__")[0] not in excluded_patterns]
        stat_feats = [f for f in STATIC_FEATURES if f in df.columns]
        if not qa_feats or not stat_feats:
            print(f"[{comp}] skipped (missing QA or STATIC features)")
            continue

        total_score = build_total_adherence_score(df, qa_feats)
        treatment = (total_score > total_score.median()).astype(int)
        if treatment.nunique() < 2:
            print(f"[{comp}] skipped (no variation in adherence split)")
            continue

        X_static = df[stat_feats].apply(pd.to_numeric, errors="coerce")
        X_static = X_static.fillna(X_static.median())
        X_static = X_static.loc[:, X_static.var() > 1e-8]

        try:
            weights, ps = compute_iptw_weights(treatment, X_static)
            balance = check_covariate_balance(X_static, treatment, weights)
            outcome = weighted_outcome_test(y, treatment, weights)
        except Exception as e:
            print(f"[{comp}] SKIPPED - IPTW failed: {e}")
            continue

        max_smd_after = balance["smd_after"].abs().max()
        results.append({
            "complication": comp,
            "n_samples": len(df),
            "n_pos": int(y.sum()),
            "treatment_coef": outcome["coef"],
            "p_value": outcome["p_value"],
            "ci_lower": outcome["ci_lower"],
            "ci_upper": outcome["ci_upper"],
            "max_smd_after_weighting": max_smd_after,
            "well_balanced": bool(max_smd_after < 0.1),
        })
        print(f"[{comp}] done | n_pos={int(y.sum())} | coef={outcome['coef']:.3f} | "
              f"p={outcome['p_value']:.4f} | max_smd_after={max_smd_after:.3f}")

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        rejected, p_adj, _, _ = multipletests(results_df["p_value"], method="fdr_bh")
        results_df["p_value_fdr"] = p_adj
        results_df["significant_fdr"] = rejected
        results_df = results_df.sort_values("p_value")
    return results_df


iptw_total_results = run_iptw_grid_total_score(
    flat_tables[48], sorted(flat_tables[48]["ComplicationName"].unique()),
    QA_FEATURES, STATIC_FEATURES, TARGET_COL
)
iptw_total_results.to_csv(RESULTS_DIR / "iptw_total_score_results.csv", index=False)
iptw_total_results


In [ ]:
def run_iptw_grid_by_pattern(
    flat_df: pd.DataFrame,
    complications_to_drill: list,
    QA_FEATURES: list,
    STATIC_FEATURES: list,
    TARGET_COL: str,
    min_positives: int = 30,
) -> pd.DataFrame:
    """
    Purpose: For each complication already flagged by the total-score IPTW grid,
             drill into individual patterns - same IPTW logic, treatment is now a
             single pattern's own average compliance score instead of the total.
             Benjamini-Hochberg FDR correction applied across all
             (complication, pattern) tests in this grid.
    """
    results = []
    for comp in tqdm(complications_to_drill, desc="IPTW: per-pattern"):
        df = flat_df[flat_df["ComplicationName"] == comp].copy()
        y = df[TARGET_COL].astype(int)
        if y.sum() < min_positives:
            continue

        excluded_patterns = QA_EXCLUDE_FOR_COMPLICATION.get(comp, [])
        qa_feats = [f for f in QA_FEATURES if f in df.columns and f.split("__")[0] not in excluded_patterns]
        stat_feats = [f for f in STATIC_FEATURES if f in df.columns]

        X_static = df[stat_feats].apply(pd.to_numeric, errors="coerce")
        X_static = X_static.fillna(X_static.median())
        X_static = X_static.loc[:, X_static.var() > 1e-8]

        patterns = sorted(set(f.split("__")[0] for f in qa_feats))
        for pattern in patterns:
            pattern_cols = [f for f in qa_feats if f.split("__")[0] == pattern]
            pattern_score = df[pattern_cols].mean(axis=1)
            treatment = (pattern_score > pattern_score.median()).astype(int)
            if treatment.nunique() < 2:
                continue

            try:
                weights, ps = compute_iptw_weights(treatment, X_static)
                balance = check_covariate_balance(X_static, treatment, weights)
                outcome = weighted_outcome_test(y, treatment, weights)
            except Exception:
                continue

            max_smd_after = balance["smd_after"].abs().max()
            results.append({
                "complication": comp,
                "pattern": pattern,
                "n_samples": len(df),
                "n_pos": int(y.sum()),
                "treatment_coef": outcome["coef"],
                "p_value": outcome["p_value"],
                "ci_lower": outcome["ci_lower"],
                "ci_upper": outcome["ci_upper"],
                "max_smd_after_weighting": max_smd_after,
                "well_balanced": bool(max_smd_after < 0.1),
            })

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        rejected, p_adj, _, _ = multipletests(results_df["p_value"], method="fdr_bh")
        results_df["p_value_fdr"] = p_adj
        results_df["significant_fdr"] = rejected
        results_df = results_df.sort_values(["complication", "p_value"])
    return results_df


significant_complications = iptw_total_results.loc[iptw_total_results["significant_fdr"], "complication"].tolist()
iptw_pattern_results = run_iptw_grid_by_pattern(
    flat_tables[48], significant_complications, QA_FEATURES, STATIC_FEATURES, TARGET_COL
)
iptw_pattern_results.to_csv(RESULTS_DIR / "iptw_pattern_results.csv", index=False)
iptw_pattern_results


In [ ]:
# ============================================================
# Manuscript-ready per-pattern IPTW table (Table: iptw-patterns)
# Significant (FDR) rows only, formatted to mirror the overall IPTW table's
# columns (beta, 95% CI, Max SMD, p_FDR). Hand this to the writing agent
# instead of the raw CSV.
# ============================================================
# Guard: the 95% CI columns come from the CI addition to run_iptw_grid_by_pattern.
# If they're absent, the per-pattern grid was run before that edit - re-run it first.
if "ci_lower" not in iptw_pattern_results.columns:
    raise RuntimeError(
        "iptw_pattern_results has no ci_lower/ci_upper - re-run the run_iptw_grid_by_pattern "
        "cell after the CI edit before building this table."
    )

# Human-readable labels reuse the matrix's dicts if they're in memory, else raw names
_comp_lbl = globals().get("COMPLICATION_LABELS", {})
_patt_lbl = globals().get("PATTERN_LABELS", {})

# Significant AND well-balanced (max SMD < 0.1). The per-pattern table must meet the
# same balance bar as the overall IPTW table; significant-but-unbalanced rows are
# positivity violations (e.g. near-universal-adherence patterns like BMI/creatinine
# measurement, where the median split is non-overlapping) and give meaningless,
# blown-up coefficients (OR ~ e^23). Drop them, but report what was dropped.
_sig = iptw_pattern_results[iptw_pattern_results["significant_fdr"]]
_kept = _sig[_sig["well_balanced"]]
_dropped = _sig[~_sig["well_balanced"]]
if len(_dropped):
    print(f"Excluded {len(_dropped)} significant-but-UNBALANCED rows (max SMD >= 0.1, not trustworthy):")
    print(_dropped[["complication", "pattern", "treatment_coef", "max_smd_after_weighting"]]
          .to_string(index=False))
    print()

iptw_patterns_table = (
    _kept
    .assign(
        Outcome=lambda d: d["complication"].map(lambda c: _comp_lbl.get(c, c)),
        Recommendation=lambda d: d["pattern"].map(lambda p: _patt_lbl.get(p, p)),
        CI=lambda d: d.apply(lambda r: f"[{r['ci_lower']:.3f}, {r['ci_upper']:.3f}]", axis=1),
    )
    .rename(columns={
        "treatment_coef": "beta",
        "max_smd_after_weighting": "max_smd",
        "p_value_fdr": "p_fdr",
    })
    .loc[:, ["Outcome", "Recommendation", "beta", "CI", "max_smd", "p_fdr", "well_balanced"]]
    .sort_values(["Outcome", "p_fdr"])
    .reset_index(drop=True)
)

# Persist the numeric version for downstream use
iptw_patterns_table.to_csv(RESULTS_DIR / "iptw_pattern_results_significant.csv", index=False)

# Rounded / p-thresholded copy purely for on-screen reading
iptw_patterns_table_display = iptw_patterns_table.copy()
iptw_patterns_table_display["beta"] = iptw_patterns_table_display["beta"].round(3)
iptw_patterns_table_display["max_smd"] = iptw_patterns_table_display["max_smd"].round(3)
iptw_patterns_table_display["p_fdr"] = iptw_patterns_table_display["p_fdr"].apply(
    lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
)

print(f"Significant + well-balanced per-pattern IPTW associations: {len(iptw_patterns_table)} rows "
      f"(max SMD < 0.1); {len(_dropped)} unbalanced row(s) excluded above.")
iptw_patterns_table_display


# Appendix S4 — Trajectory Shape of Adherence Scores Within an Admission

Supplementary analysis for the JAMIA paper. Three questions:
- **S4.1**  Does a mean of 0.80 reflect a flat trajectory (every day ≈ 0.80) or a drop (every day = 1.00 except one at 0.20)?
- **S4.2**  Is adherence better early or late in an admission?
- **S4.3**  Does admission length shape either answer?

Operates on the fourteen QA patterns (same as §S3 of the paper's supplementary).
Inputs: the already-loaded `qa_df` (per-event compliance scores keyed on `VisitId`, with `StartDateTime` and `ComplianceScore`) and `visit_master` (per-admission `StartDateTime` / `EndDateTime`).

Nothing here is committed to the manuscript yet; this cell block only produces the numbers and plots that will populate Appendix S4 if we decide to include it.

In [ ]:
# ------------------------------------------------------------------
# S4 inputs and helper -- load cached data in this cell
# ------------------------------------------------------------------
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

MAX_DAY = 14
DB_PATH = Path("backend/data/mediator.db")
EXTERNAL_DATA_DIR = Path("../Data Exploration")

# Resolve paths whether Jupyter starts in Mediator/ or Mediator/analysis/.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "analysis":
    REPO_ROOT = REPO_ROOT.parent

db_path = (REPO_ROOT / DB_PATH).resolve()
cache_dir = (REPO_ROOT / EXTERNAL_DATA_DIR).resolve()
visits_path = cache_dir / "visits_master.csv"
cohort_path = cache_dir / "context_static_input.csv"

for path in (db_path, visits_path, cohort_path):
    if not path.exists():
        raise FileNotFoundError(f"Required cached input not found: {path}")

# Read QA scores from the existing Mediator database.
with sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True) as conn:
    qa_df = pd.read_sql_query(
        """
        SELECT PatientId, PatternName, StartDateTime, ComplianceScore
        FROM PatientQAScores
        """,
        conn,
    )

# In this database, PatientId is the admission/visit ID.
qa_df = qa_df.rename(columns={"PatientId": "VisitId"})

# Load visit dates and the cohort IDs used by the main analysis.
visit_master = pd.read_csv(visits_path, low_memory=False).rename(
    columns={
        "visit_id": "VisitId",
        "start_datetime": "StartDateTime",
        "end_datetime": "EndDateTime",
    }
)
cohort_ids = pd.read_csv(cohort_path, usecols=["visit_id"])["visit_id"].dropna().unique()

# Match the main analysis cohort and exclude ambiguous visit IDs.
if "person_id" in visit_master.columns:
    people_per_visit = visit_master.groupby("VisitId")["person_id"].nunique()
    invalid_ids = people_per_visit[people_per_visit > 1].index
    visit_master = visit_master[~visit_master["VisitId"].isin(invalid_ids)]

visit_master = (
    visit_master[visit_master["VisitId"].isin(cohort_ids)]
    .sort_values("StartDateTime")
    .drop_duplicates(subset=["VisitId"], keep="last")
    [["VisitId", "StartDateTime", "EndDateTime"] +
     (["hospital"] if "hospital" in visit_master.columns else [])]
    .copy()
)

print(
    f"Loaded {len(qa_df):,} QA rows and "
    f"{visit_master['VisitId'].nunique():,} cohort visits"
)


def build_daily_adherence(
    qa_df: pd.DataFrame,
    visit_master: pd.DataFrame,
) -> pd.DataFrame:
    """Return one row per visit, pattern, and day with a daily mean score."""
    qa = qa_df[
        ["VisitId", "PatternName", "StartDateTime", "ComplianceScore"]
    ].copy()
    vm = visit_master.rename(
        columns={
            "StartDateTime": "adm_start",
            "EndDateTime": "adm_end",
        }
    ).copy()

    for col in ("StartDateTime", "adm_start", "adm_end"):
        if col in qa.columns:
            qa[col] = pd.to_datetime(qa[col], errors="coerce", utc=True)
        if col in vm.columns:
            vm[col] = pd.to_datetime(vm[col], errors="coerce", utc=True)

    merged = qa.merge(vm, on="VisitId", how="inner")
    merged = merged.dropna(subset=["StartDateTime", "adm_start"])
    merged["ComplianceScore"] = pd.to_numeric(
        merged["ComplianceScore"], errors="coerce"
    )
    merged = merged.dropna(subset=["ComplianceScore"])

    hours = (merged["StartDateTime"] - merged["adm_start"]).dt.total_seconds() / 3600
    merged["day_of_admission"] = (hours // 24).astype(int) + 1
    merged = merged[merged["day_of_admission"].between(1, MAX_DAY)]

    daily = (
        merged.groupby(
            ["VisitId", "PatternName", "day_of_admission"],
            as_index=False,
        )["ComplianceScore"]
        .mean()
        .rename(columns={"ComplianceScore": "daily_score"})
    )

    vm["los_days"] = (
        (vm["adm_end"] - vm["adm_start"]).dt.total_seconds() / 86400
    ).clip(lower=0.5, upper=MAX_DAY)

    return daily.merge(vm[["VisitId", "los_days"]], on="VisitId", how="left")


daily_df = build_daily_adherence(qa_df, visit_master)
print(
    f"daily_df: {len(daily_df):,} rows across "
    f"{daily_df['VisitId'].nunique():,} admissions and "
    f"{daily_df['PatternName'].nunique()} patterns"
)
daily_df.head()


## S4.1 — Shape: typical day vs extreme pull

For each `(VisitId, PatternName)` with at least 3 daily scores we compute:
- `mean_daily`  — mean of daily scores (what the main table shows, roughly)
- `std_daily`   — within-admission SD
- `min_daily`   — the worst day
- `pull_ratio = min_daily / mean_daily`  — close to 1 if flat, close to 0 if a single day pulled the mean down

An admission with pull_ratio ≈ 1 is "typical day = mean day". pull_ratio < 0.6 flags admissions where the mean is substantially pulled by one or two outlier days.

In [ ]:
# ------------------------------------------------------------------
# S4.1  Per-(VisitId, Pattern) shape statistics
# ------------------------------------------------------------------
MIN_DAYS_FOR_SHAPE = 3

shape_stats = (
    daily_df
    .groupby(['VisitId', 'PatternName'])
    .agg(n_days=('daily_score', 'size'),
         mean_daily=('daily_score', 'mean'),
         std_daily=('daily_score', 'std'),
         min_daily=('daily_score', 'min'))
    .reset_index()
)
shape_stats = shape_stats[shape_stats['n_days'] >= MIN_DAYS_FOR_SHAPE].copy()
shape_stats['pull_ratio'] = (shape_stats['min_daily'] /
                              shape_stats['mean_daily'].replace(0, np.nan))

# per-pattern summary -- what will populate Table S3
pattern_shape_summary = (
    shape_stats
    .groupby('PatternName')
    .agg(n_admissions=('VisitId', 'nunique'),
         median_std=('std_daily', 'median'),
         median_pull_ratio=('pull_ratio', 'median'),
         share_pull_below_06=('pull_ratio', lambda s: (s < 0.6).mean()))
    .reset_index()
    .sort_values('share_pull_below_06', ascending=False)
)

pattern_shape_summary.to_csv(RESULTS_DIR / 'appendix_s4_table_s3_pattern_shape_summary.csv', index=False)
pattern_shape_summary.round(3)


In [ ]:
# ------------------------------------------------------------------
# S4.1b -- are trajectory shapes different across hospitals?
# ------------------------------------------------------------------
if "hospital" not in visit_master.columns:
    raise KeyError("visits_master.csv must include a 'hospital' column for this analysis")

hospital_map = visit_master[["VisitId", "hospital"]].drop_duplicates()
if hospital_map["VisitId"].duplicated().any():
    raise ValueError("A VisitId maps to more than one hospital in visit_master")

site_shape = shape_stats.merge(hospital_map, on="VisitId", how="inner", validate="many_to_one")
site_shape["ratio_defined"] = site_shape["pull_ratio"].notna()
site_shape["nearly_flat"] = site_shape["pull_ratio"] >= 0.9
site_shape["pulled_by_low_day"] = site_shape["pull_ratio"] < 0.6

# Hospital summaries use the same admission-pattern pairs as shape_stats.
hospital_shape_summary = (
    site_shape.groupby("hospital")
    .agg(
        n_admission_pattern_pairs=("VisitId", "size"),
        n_admissions=("VisitId", "nunique"),
        n_defined_ratios=("ratio_defined", "sum"),
        share_nearly_flat=("nearly_flat", "mean"),
        share_pulled_by_low_day=("pulled_by_low_day", "mean"),
        median_within_admission_sd=("std_daily", "median"),
    )
    .reset_index()
    .sort_values("hospital")
)
hospital_shape_summary[["share_nearly_flat", "share_pulled_by_low_day"]] *= 100

# Pattern-stratified summaries separate site differences from different mixes
# of recommendations or patterns being applicable at each hospital.
hospital_pattern_summary = (
    site_shape.groupby(["hospital", "PatternName"])
    .agg(
        n_admission_pattern_pairs=("VisitId", "size"),
        n_defined_ratios=("ratio_defined", "sum"),
        share_nearly_flat=("nearly_flat", "mean"),
        share_pulled_by_low_day=("pulled_by_low_day", "mean"),
    )
    .reset_index()
)
hospital_pattern_summary[["share_nearly_flat", "share_pulled_by_low_day"]] *= 100

print("Hospital-level trajectory summaries (percentages):")
display(hospital_shape_summary.round(1))
print("Pattern-stratified hospital summaries (percentages; inspect denominators):")
display(hospital_pattern_summary.sort_values(["PatternName", "hospital"]).round(1))


In [ ]:
# ------------------------------------------------------------------
# S4.1  Figure S1 -- shape scatter + histogram
# ------------------------------------------------------------------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# -- left: pull_ratio vs mean_daily, one dot per admission x pattern
ax = axes[0]
ax.scatter(shape_stats['mean_daily'], shape_stats['pull_ratio'],
           s=3, alpha=0.08, color='#2c3e50', rasterized=True)
ax.plot([0, 1], [0, 1], ls='--', lw=0.8, color='k', alpha=0.4,
        label='pull_ratio = mean (worst day == 0)')
ax.axhline(1.0, ls='--', lw=0.8, color='k', alpha=0.4,
           label='pull_ratio = 1 (flat trajectory)')
ax.set_xlabel('Mean daily score')
ax.set_ylabel('pull_ratio = min_daily / mean_daily')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.set_title('Where the mean sits relative to the worst day')
ax.legend(loc='lower right', fontsize=8)

# -- right: pull_ratio histogram within three mean-score bins
ax = axes[1]
BINS = [(0.3, 0.6, 'mean 0.3-0.6'),
        (0.6, 0.8, 'mean 0.6-0.8'),
        (0.8, 1.0, 'mean 0.8-1.0')]
colors = ['#e74c3c', '#f39c12', '#27ae60']
for (lo, hi, lab), color in zip(BINS, colors):
    sub = shape_stats[(shape_stats['mean_daily'] >= lo) &
                      (shape_stats['mean_daily'] <  hi)]
    ax.hist(sub['pull_ratio'].dropna(),
            bins=np.linspace(0, 1, 21), alpha=0.5,
            color=color, label=f'{lab}  (n={len(sub):,})',
            density=True)
ax.set_xlabel('pull_ratio')
ax.set_ylabel('density')
ax.set_title('Pull-ratio distribution by mean-score band')
ax.legend(fontsize=8)

fig.suptitle('Appendix S4.1 — Trajectory shape within admissions', y=1.02)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'appendix_s4_fig_s1_shape.png', dpi=200, bbox_inches='tight')
fig.savefig(RESULTS_DIR / 'appendix_s4_fig_s1_shape.pdf', bbox_inches='tight')
fig


## S4.2 — Timing: early vs late within admission

Mean daily score across all contributing admissions, by day of admission (day 1 → 14), per pattern.
A sample-size curve underlays each pattern's trajectory so the reader can see how many admissions are contributing to each late-day data point.

A **drift-down** curve on routine insulin recommendations would make the de-escalation effect you discuss in Limitations directly visible.
A **flat** curve means the aggregate mean is a fair summary of the trajectory.

In [ ]:
# ------------------------------------------------------------------
# S4.2  Day-of-admission mean per pattern + contributing n
# ------------------------------------------------------------------
by_day = (
    daily_df
    .groupby(['PatternName', 'day_of_admission'])
    .agg(mean_score=('daily_score', 'mean'),
         n_contributing=('VisitId', 'nunique'))
    .reset_index()
)
by_day.to_csv(RESULTS_DIR / 'appendix_s4_timing_by_day.csv', index=False)

patterns = sorted(by_day['PatternName'].unique())
n = len(patterns)
cols = 4
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4.0 * cols, 2.4 * rows), squeeze=False)

for ax, pat in zip(axes.flat, patterns):
    sub = by_day[by_day['PatternName'] == pat].sort_values('day_of_admission')
    overall_mean = daily_df.loc[daily_df['PatternName'] == pat, 'daily_score'].mean()

    ax_bar = ax.twinx()
    ax_bar.bar(sub['day_of_admission'], sub['n_contributing'],
               color='#bdc3c7', alpha=0.4, width=0.8)
    ax_bar.set_yticks([])

    ax.plot(sub['day_of_admission'], sub['mean_score'],
            marker='o', ms=3, lw=1.3, color='#2c3e50')
    ax.axhline(overall_mean, ls=':', lw=0.8, color='#c0392b',
               label=f'overall mean = {overall_mean:.2f}')

    label = PATTERN_LABELS.get(pat, pat) if 'PATTERN_LABELS' in globals() else pat
    ax.set_title(label, fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0.5, MAX_DAY + 0.5)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, loc='lower left')

for ax in axes.flat[n:]:
    ax.axis('off')

fig.suptitle('Appendix S4.2 — Daily adherence by day of admission (grey bars: n contributing)',
             y=1.00)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'appendix_s4_fig_s2_timing.png', dpi=200, bbox_inches='tight')
fig.savefig(RESULTS_DIR / 'appendix_s4_fig_s2_timing.pdf', bbox_inches='tight')
fig


## S4.3 — Length: does duration shape the trajectory?

Two scatter plots over admissions:
- *Left:*  mean per-admission adherence vs length of stay
- *Right:* within-admission SD of daily scores vs length of stay

If longer admissions have systematically lower mean adherence or higher variability, that calls for comment in the Discussion about how the length distribution of the groups being compared behaves.

In [ ]:
# ------------------------------------------------------------------
# S4.3  Length vs mean + length vs within-admission SD
# ------------------------------------------------------------------
per_adm = (
    daily_df
    .groupby('VisitId')
    .agg(mean_adherence=('daily_score', 'mean'),
         std_adherence=('daily_score', 'std'),
         los_days=('los_days', 'first'),
         n_days=('day_of_admission', 'nunique'))
    .reset_index()
)
# at least two days to have a meaningful SD
per_adm_sd = per_adm.dropna(subset=['std_adherence'])
per_adm.to_csv(RESULTS_DIR / 'appendix_s4_per_admission_shape.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# bin by integer length of stay for the binned-mean overlay
bin_edges = np.arange(1, MAX_DAY + 2) - 0.5
bin_mids  = np.arange(1, MAX_DAY + 1)


def binned_mean(x, y):
    y = np.asarray(y)
    x = np.asarray(x)
    out = np.full_like(bin_mids, np.nan, dtype=float)
    for i, m in enumerate(bin_mids):
        mask = (x >= bin_edges[i]) & (x < bin_edges[i+1])
        if mask.sum() >= 20:
            out[i] = y[mask].mean()
    return out


# -- left: mean adherence vs LOS
ax = axes[0]
ax.scatter(per_adm['los_days'], per_adm['mean_adherence'],
           s=3, alpha=0.08, color='#2c3e50', rasterized=True)
bm = binned_mean(per_adm['los_days'], per_adm['mean_adherence'])
ax.plot(bin_mids, bm, 'o-', color='#c0392b', ms=5,
        label='binned mean (bins with n >= 20)')
ax.set_xlabel('Length of stay (days, capped at 14)')
ax.set_ylabel('Mean per-admission adherence')
ax.set_xlim(0, MAX_DAY + 1)
ax.set_ylim(0, 1.05)
ax.set_title('Mean adherence vs admission length')
ax.legend(fontsize=8, loc='lower right')

# -- right: within-admission SD vs LOS
ax = axes[1]
ax.scatter(per_adm_sd['los_days'], per_adm_sd['std_adherence'],
           s=3, alpha=0.08, color='#2c3e50', rasterized=True)
bm = binned_mean(per_adm_sd['los_days'], per_adm_sd['std_adherence'])
ax.plot(bin_mids, bm, 'o-', color='#c0392b', ms=5,
        label='binned mean (bins with n >= 20)')
ax.set_xlabel('Length of stay (days, capped at 14)')
ax.set_ylabel('Within-admission SD of daily scores')
ax.set_xlim(0, MAX_DAY + 1)
ax.set_title('Trajectory variability vs admission length')
ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Appendix S4.3 — Admission length vs trajectory shape', y=1.02)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'appendix_s4_fig_s3_length.png', dpi=200, bbox_inches='tight')
fig.savefig(RESULTS_DIR / 'appendix_s4_fig_s3_length.pdf', bbox_inches='tight')
fig


## S4 — summary numbers for the appendix text

A compact print-out whose output should populate the opening paragraph of §S4
(share of flat-trajectory admissions, share of pulled-mean admissions, correlation
of length with mean adherence).

In [ ]:
# ------------------------------------------------------------------
# S4  summary numbers to quote in appendix text
# ------------------------------------------------------------------
from scipy.stats import spearmanr

print('Trajectory-shape summary (shape_stats, admissions with n_days >= 3):')
print(f'  admissions x pattern pairs: {len(shape_stats):,}')
print(f'  share with pull_ratio >= 0.9 (nearly flat):'
      f' {(shape_stats["pull_ratio"] >= 0.9).mean():.1%}')
print(f'  share with pull_ratio <  0.6 (pulled by extremes):'
      f' {(shape_stats["pull_ratio"] <  0.6).mean():.1%}')
print()

print('Length-shape relationships (per_adm, Spearman rho):')
r1, p1 = spearmanr(per_adm['los_days'], per_adm['mean_adherence'])
print(f'  length vs mean adherence : rho={r1:+.3f}  p={p1:.2e}')
r2, p2 = spearmanr(per_adm_sd['los_days'], per_adm_sd['std_adherence'])
print(f'  length vs within-adm SD  : rho={r2:+.3f}  p={p2:.2e}')
print()

# early vs late slope per pattern
early_late = (daily_df.assign(third=lambda d: np.where(d['day_of_admission'] <= 3, 'early',
                                                       np.where(d['day_of_admission'] >= 8, 'late', 'mid')))
                     .groupby(['PatternName', 'third'])['daily_score']
                     .mean()
                     .unstack('third'))
early_late['late_minus_early'] = early_late['late'] - early_late['early']
print('Per-pattern early (day 1-3) vs late (day 8+) adherence:')
print(early_late.round(3).sort_values('late_minus_early'))


# Abstractions: threshold proximity and abstraction stability

Existing database access is read-only. Direct numeric inputs contributing states or trends are evaluated in the first 48 hours. Derived numeric concepts and downstream predictions are excluded. Generated outputs go to `backend/data/analysis/`, outside this source directory. Use the Mediator virtual-environment kernel.


## Resolve the TAK tree and verify the database export

The comparison checks every input record, rather than counts or a sample.


In [ ]:
run_analysis('threshold-proximity', 'export_repository.py')
run_analysis('threshold-proximity', 'validate_database.py')


## Full-cohort proximity to state and trend thresholds

Bands are normalized by local cutoff spacing or the trend-change threshold; they are not assay-error estimates.


In [ ]:
run_analysis('threshold-proximity', 'validate_trends.py')
run_analysis('threshold-proximity', 'analyze.py')
run_analysis('threshold-proximity', 'report.py')


In [ ]:
display(pd.read_csv(ANALYSIS_RESULTS / 'threshold-proximity/state_summary.csv'))
display(pd.read_csv(ANALYSIS_RESULTS / 'threshold-proximity/trend_summary.csv'))


## Sampled Mediator stability under perturbations and observation removal

500 stratified admissions and two realizations per condition. Actual state/trend implementations include persistence, merging, regression, and clipping. Script execution supports Windows worker processes.


In [ ]:
run_analysis('mediator-stability', 'run_study.py', '--patients', 500)
run_analysis('mediator-stability', 'summarize.py')


In [ ]:
display(pd.read_csv(ANALYSIS_RESULTS / 'mediator-stability/results.csv'))
display(Markdown((ANALYSIS_RESULTS / 'mediator-stability/report.md').read_text(encoding='utf-8')))
